# < Model 1. baseline >
category, 
amt, 
trans_hour, 
age

: 별도의 개인화.행동 파생변수 없이 현재 거래의 기본 정보만 사용한 모델>

- 먼저 Baseline 70:30 모델에서 공통 고정 임계값을 한 번 결정

In [2]:
%pip install lightgbm

  Using cached narwhals-2.24.0-py3-none-any.whl.metadata (15 kB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   -- ------------------------------------- 0.1/1.4 MB 919.0 kB/s eta 0:00:02
   ---------- ----------------------------- 0.4/1.4 MB 2.9 MB/s eta 0:00:01
   ------------------------------------- -- 1.3/1.4 MB 7.4 MB/s eta 0:00:01
   ---------------------------------------  1.4/1.4 MB 7.1 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 5.1 MB/s eta 0:00:00
Using cached narwhals-2.24.0-py3-none-any.whl (461 kB)
Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl (36.6 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import lightgbm as lgb

print("LightGBM 버전:", lgb.__version__)

LightGBM 버전: 4.7.0


In [2]:
%pip install scikit-learn

  Using cached scikit_learn-1.9.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.9.0-cp311-cp311-win_amd64.whl (8.3 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import sklearn
print("scikit-learn 버전:", sklearn.__version__)

scikit-learn 버전: 1.9.0


In [1]:
import lightgbm
import sklearn

print(lightgbm.__version__)
print(sklearn.__version__)

4.7.0
1.9.0


In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve
)


# =========================================================
# 1. 데이터 불러오기
# =========================================================
csv_path = (
    r"C:\Users\splen\OneDrive\Desktop\BDAI_"
    r"\BOOSTMAP\Fraud-FDS-Project\data"
    r"\fraud_full_features.csv"
)

df = pd.read_csv(
    csv_path,
    parse_dates=["trans_date_trans_time"]
)

df = (
    df.sort_values("trans_date_trans_time")
      .reset_index(drop=True)
)

print("전체 데이터:", df.shape)


# =========================================================
# 2. Baseline 설명변수
# =========================================================
baseline_features = [
    "category",
    "amt",
    "trans_hour",
    "age"
]

target = "is_fraud"

X = df[baseline_features].copy()
y = df[target].astype("int8").copy()

# LightGBM 범주형 변수 지정
X["category"] = X["category"].astype("category")


# =========================================================
# 3. 시간순 70:30 분할
# =========================================================
split_index = int(len(df) * 0.70)

X_train = X.iloc[:split_index].copy()
y_train = y.iloc[:split_index].copy()

X_valid = X.iloc[split_index:].copy()
y_valid = y.iloc[split_index:].copy()

print("\nTrain:", X_train.shape)
print("Valid:", X_valid.shape)

print(
    "Train 기간:",
    df.loc[:split_index - 1, "trans_date_trans_time"].min(),
    "~",
    df.loc[:split_index - 1, "trans_date_trans_time"].max()
)

print(
    "Valid 기간:",
    df.loc[split_index:, "trans_date_trans_time"].min(),
    "~",
    df.loc[split_index:, "trans_date_trans_time"].max()
)

print("\nTrain 이상거래 건수:", int(y_train.sum()))
print("Valid 이상거래 건수:", int(y_valid.sum()))

print("Train 이상거래율:", y_train.mean())
print("Valid 이상거래율:", y_valid.mean())


# =========================================================
# 4. 클래스 불균형 가중치
# =========================================================
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())

scale_pos_weight = (
    negative_count / positive_count
)

print("\n정상거래 수:", negative_count)
print("이상거래 수:", positive_count)
print("scale_pos_weight:", scale_pos_weight)


# =========================================================
# 5. 모델 비교용 공통 하이퍼파라미터
# =========================================================
MODEL_PARAMS = {
    "objective": "binary",
    "boosting_type": "gbdt",

    "n_estimators": 1000,
    "learning_rate": 0.05,

    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 50,

    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,

    "reg_alpha": 0.0,
    "reg_lambda": 0.0,

    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1
}

baseline_model_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight
)


# =========================================================
# 6. 학습
# LightGBM 4.7.0 방식으로 eval_X, eval_y 사용
# =========================================================
baseline_model_7030.fit(
    X_train,
    y_train,

    eval_X=X_valid,
    eval_y=y_valid,

    eval_metric="average_precision",
    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# =========================================================
# 7. 검증셋 예측확률
# =========================================================
valid_prob = baseline_model_7030.predict_proba(
    X_valid,
    num_iteration=baseline_model_7030.best_iteration_
)[:, 1]


# =========================================================
# 8. Baseline 70:30에서 공통 임계값 1회 선정
# =========================================================
precisions, recalls, thresholds = precision_recall_curve(
    y_valid,
    valid_prob
)

f1_scores = (
    2 * precisions[:-1] * recalls[:-1]
    / (
        precisions[:-1]
        + recalls[:-1]
        + 1e-12
    )
)

best_index = int(np.argmax(f1_scores))

FIXED_THRESHOLD = float(
    thresholds[best_index]
)

valid_pred = (
    valid_prob >= FIXED_THRESHOLD
).astype("int8")


# =========================================================
# 9. 성능 계산
# =========================================================
result_7030 = {
    "model": "Baseline",
    "split": "70:30",
    "best_iteration": baseline_model_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid,
        valid_prob
    ),

    "roc_auc": roc_auc_score(
        y_valid,
        valid_prob
    ),

    "precision": precision_score(
        y_valid,
        valid_pred,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid,
        valid_pred,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid,
        valid_pred,
        zero_division=0
    )
}


# =========================================================
# 10. 결과 출력
# =========================================================
print("\n========== Baseline 70:30 재실험 ==========")

print(
    "Best iteration :",
    result_7030["best_iteration"]
)

print(
    "고정 임계값     :",
    round(result_7030["threshold"], 6)
)

print(
    "PR-AUC         :",
    round(result_7030["pr_auc"], 6)
)

print(
    "ROC-AUC        :",
    round(result_7030["roc_auc"], 6)
)

print(
    "Precision      :",
    round(result_7030["precision"], 6)
)

print(
    "Recall         :",
    round(result_7030["recall"], 6)
)

print(
    "F1-score       :",
    round(result_7030["f1"], 6)
)

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid,
        valid_pred
    )
)

print("\n공통 하이퍼파라미터")
for key, value in MODEL_PARAMS.items():
    print(f"{key}: {value}")

print(
    "scale_pos_weight:",
    scale_pos_weight
)

전체 데이터: (1296675, 31)

Train: (907672, 4)
Valid: (389003, 4)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.0061310581152330445

정상거래 수: 902551
이상거래 수: 5121
scale_pos_weight: 176.24506932239797
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.867393	valid_0's binary_logloss: 0.0598627
[100]	valid_0's average_precision: 0.88679	valid_0's binary_logloss: 0.0490198
[150]	valid_0's average_precision: 0.890785	valid_0's binary_logloss: 0.0433696
[200]	valid_0's average_precision: 0.895938	valid_0's binary_logloss: 0.0394014
[250]	valid_0's average_precision: 0.898471	valid_0's binary_logloss: 0.0365787
[300]	valid_0's average_precision: 0.899856	valid_0's binary_logloss: 0.0342252
[350]	valid_0's average_precision: 0.900194	valid_0's binary_logloss: 0.0325393
[400]	valid_0's average_precision: 0.90

(907672, 4)에서 4는 설명변수 개수.

[50] <- 50번째 트리까지 학습.

early_stopping으로 450개 트리까지 학습, best iteration은 403번째 트리.

Confusion Matrix를 보면,

실제 이상거래 2,385(483 + 1902)건 중, 1902건 탐지, 483건 미탐지

정상거래 중 226건을 이상거래로 오탐

- 80:20 비교 (임계값 0.9900239로 쓰기, scale_pos_weight만 80% 학습셋 기준으로 재계산)

In [5]:
# =========================================================
# Baseline 80:20 비교
# =========================================================

FIXED_THRESHOLD = 0.990239

# 1. 시간순 80:20 분할
split_index_8020 = int(len(df) * 0.80)

X_train_8020 = X.iloc[:split_index_8020].copy()
y_train_8020 = y.iloc[:split_index_8020].copy()

X_valid_8020 = X.iloc[split_index_8020:].copy()
y_valid_8020 = y.iloc[split_index_8020:].copy()

print("Train:", X_train_8020.shape)
print("Valid:", X_valid_8020.shape)

print(
    "Train 기간:",
    df.loc[:split_index_8020 - 1, "trans_date_trans_time"].min(),
    "~",
    df.loc[:split_index_8020 - 1, "trans_date_trans_time"].max()
)

print(
    "Valid 기간:",
    df.loc[split_index_8020:, "trans_date_trans_time"].min(),
    "~",
    df.loc[split_index_8020:, "trans_date_trans_time"].max()
)

print("\nTrain 이상거래 건수:", int(y_train_8020.sum()))
print("Valid 이상거래 건수:", int(y_valid_8020.sum()))

print("Train 이상거래율:", y_train_8020.mean())
print("Valid 이상거래율:", y_valid_8020.mean())


# 2. 80% 학습셋 기준 scale_pos_weight 계산
negative_count_8020 = int((y_train_8020 == 0).sum())
positive_count_8020 = int((y_train_8020 == 1).sum())

scale_pos_weight_8020 = (
    negative_count_8020 / positive_count_8020
)

print("\n정상거래 수:", negative_count_8020)
print("이상거래 수:", positive_count_8020)
print("scale_pos_weight:", scale_pos_weight_8020)


# 3. 동일 하이퍼파라미터 모델 생성
baseline_model_8020 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_8020
)


# 4. 학습
baseline_model_8020.fit(
    X_train_8020,
    y_train_8020,

    eval_X=X_valid_8020,
    eval_y=y_valid_8020,

    eval_metric="average_precision",
    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 5. 예측확률
valid_prob_8020 = baseline_model_8020.predict_proba(
    X_valid_8020,
    num_iteration=baseline_model_8020.best_iteration_
)[:, 1]


# 6. 70:30에서 정한 고정 임계값 적용
valid_pred_8020 = (
    valid_prob_8020 >= FIXED_THRESHOLD
).astype("int8")


# 7. 평가
result_8020 = {
    "model": "Baseline",
    "split": "80:20",
    "best_iteration": baseline_model_8020.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_8020,
        valid_prob_8020
    ),

    "roc_auc": roc_auc_score(
        y_valid_8020,
        valid_prob_8020
    ),

    "precision": precision_score(
        y_valid_8020,
        valid_pred_8020,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_8020,
        valid_pred_8020,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_8020,
        valid_pred_8020,
        zero_division=0
    )
}


# 8. 결과 출력
print("\n========== Baseline 80:20 ==========")

print(
    "Best iteration :",
    result_8020["best_iteration"]
)

print(
    "고정 임계값     :",
    round(result_8020["threshold"], 6)
)

print(
    "PR-AUC         :",
    round(result_8020["pr_auc"], 6)
)

print(
    "ROC-AUC        :",
    round(result_8020["roc_auc"], 6)
)

print(
    "Precision      :",
    round(result_8020["precision"], 6)
)

print(
    "Recall         :",
    round(result_8020["recall"], 6)
)

print(
    "F1-score       :",
    round(result_8020["f1"], 6)
)

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_8020,
        valid_pred_8020
    )
)

Train: (1037340, 4)
Valid: (259335, 4)
Train 기간: 2019-01-01 00:00:18 ~ 2020-03-06 07:15:17
Valid 기간: 2020-03-06 07:16:43 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5968
Valid 이상거래 건수: 1538
Train 이상거래율: 0.005753176393467908
Valid 이상거래율: 0.005930553145545337

정상거래 수: 1031372
이상거래 수: 5968
scale_pos_weight: 172.8170241286863
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.874486	valid_0's binary_logloss: 0.0587517
[100]	valid_0's average_precision: 0.887648	valid_0's binary_logloss: 0.0486915
[150]	valid_0's average_precision: 0.892842	valid_0's binary_logloss: 0.0432684
[200]	valid_0's average_precision: 0.8968	valid_0's binary_logloss: 0.0395949
[250]	valid_0's average_precision: 0.898557	valid_0's binary_logloss: 0.0370391
[300]	valid_0's average_precision: 0.89888	valid_0's binary_logloss: 0.0349521
[350]	valid_0's average_precision: 0.900097	valid_0's binary_logloss: 0.0332299
[400]	valid_0's average_precision: 0.899874	valid_0's binary_logl

결과를 비교하면, baseline 모델은 70:30 분할 비율이 성능이 더 좋음.

이제부터 모든 모델을 두 번씩 돌리지 않고 70:30 한 번씩만 돌리기.

상위 2~3개 조합이 정해졌을 때 expanding window로 여러 미래 구간에서 안정성 확인.

# Model 2: 이진변수만 사용

- is_high_amt      : 거래금액이 500 이상인지
- high_speed       : 이동속도가 100 이상인지
- is_online        : 온라인 업종 거래인지
- risk_time_22_04  : 밤 10시~새벽 4시 거래인지

이 모델은 원본 변수를 이진 변수로 압축했을 때도 성능이 유지되는지 확인하는 모델

In [6]:
# =========================================================
# Model 2: 규칙형 변수 4개
# 시간순 70:30
# =========================================================

MODEL_NAME = "Rule-based only"
FIXED_THRESHOLD = 0.990239

model2_features = [
    "is_online",
    "is_high_amt",
    "risk_time_22_04",
    "high_speed"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model2_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model2_features))
print("사용 변수:", model2_features)


# 2. 설명변수와 목표변수 생성
X_model2 = df[model2_features].copy()
y_model2 = df["is_fraud"].astype("int8").copy()

print("\n변수 자료형")
print(X_model2.dtypes)

print("\n결측치 수")
print(X_model2.isna().sum())


# 3. 시간순 70:30 분할
split_index_model2 = int(len(df) * 0.70)

X_train_model2 = X_model2.iloc[:split_index_model2].copy()
X_valid_model2 = X_model2.iloc[split_index_model2:].copy()

y_train_model2 = y_model2.iloc[:split_index_model2].copy()
y_valid_model2 = y_model2.iloc[split_index_model2:].copy()

print("\nTrain:", X_train_model2.shape)
print("Valid:", X_valid_model2.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model2 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model2 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model2:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model2:,
        "trans_date_trans_time"
    ].max()
)

print("\nTrain 이상거래 건수:", int(y_train_model2.sum()))
print("Valid 이상거래 건수:", int(y_valid_model2.sum()))

print("Train 이상거래율:", y_train_model2.mean())
print("Valid 이상거래율:", y_valid_model2.mean())


# 4. 클래스 불균형 가중치
negative_count_model2 = int((y_train_model2 == 0).sum())
positive_count_model2 = int((y_train_model2 == 1).sum())

scale_pos_weight_model2 = (
    negative_count_model2
    / positive_count_model2
)

print("\n정상거래 수:", negative_count_model2)
print("이상거래 수:", positive_count_model2)
print("scale_pos_weight:", scale_pos_weight_model2)


# 5. Baseline과 동일한 하이퍼파라미터로 모델 생성
model2_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model2
)


# 6. 학습
model2_7030.fit(
    X_train_model2,
    y_train_model2,

    eval_X=X_valid_model2,
    eval_y=y_valid_model2,

    eval_metric="average_precision",

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model2 = model2_7030.predict_proba(
    X_valid_model2,
    num_iteration=model2_7030.best_iteration_
)[:, 1]


# 8. Baseline에서 정한 고정 임계값 적용
valid_pred_model2 = (
    valid_prob_model2 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model2 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model2_features),
    "best_iteration": model2_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model2,
        valid_prob_model2
    ),

    "roc_auc": roc_auc_score(
        y_valid_model2,
        valid_prob_model2
    ),

    "precision": precision_score(
        y_valid_model2,
        valid_pred_model2,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model2,
        valid_pred_model2,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model2,
        valid_pred_model2,
        zero_division=0
    )
}


# 10. 결과 출력
print("\n========== Model 2: Rule-based only 70:30 ==========")

print("변수 수         :", result_model2["feature_count"])
print("Best iteration :", result_model2["best_iteration"])
print("고정 임계값     :", round(result_model2["threshold"], 6))
print("PR-AUC         :", round(result_model2["pr_auc"], 6))
print("ROC-AUC        :", round(result_model2["roc_auc"], 6))
print("Precision      :", round(result_model2["precision"], 6))
print("Recall         :", round(result_model2["recall"], 6))
print("F1-score       :", round(result_model2["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model2,
        valid_pred_model2
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model2["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model2["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model2["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model2["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model2["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 4
사용 변수: ['is_online', 'is_high_amt', 'risk_time_22_04', 'high_speed']

변수 자료형
is_online          int64
is_high_amt        int64
risk_time_22_04    int64
high_speed         int64
dtype: object

결측치 수
is_online          0
is_high_amt        0
risk_time_22_04    0
high_speed         0
dtype: int64

Train: (907672, 4)
Valid: (389003, 4)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.0061310581152330445

정상거래 수: 902551
이상거래 수: 5121
scale_pos_weight: 176.24506932239797
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.328726	valid_0's binary_logloss: 0.320069
Early stopping, best iteration is:
[2]	valid_0's average_precision: 0.329021	valid_0's binary_logloss: 0.284655
Evaluated only: average_precision

========== Model 2: Rule-based only 70:30 ==========
변수 수         : 4
Best iteration : 2

1. PR-AUC = 0.33 정도로, Baseline의 0.90보다 훨씬 낮음.

=> 이진 변수만 써서 원래 정보가 너무 많이 사라진 것.

2. Precision·Recall·F1이 모두 0 : 고정 임계값 0.990239를 넘는 예측확률이 하나도 없었기 때문.

3. confusion matrix도 모든 거래를 정상으로 분류함.

4. Best iteration=2 : 같은 맥락. 변수 조합이 단순해서 초반 이후 검증 PR-AUC가 거의 개선되지 않은 것.

=> 이 실험의 의미:

규칙형 "이진 변수만"으로는 원본 변수의 세부 정보를 충분히 대체하기 어렵다

# Model 3: Baseline + 이진변수

총 8가지 설명변수

이 모델은 원본 정보와 이진 정보를 함께 줬을 때 실제 성능이 추가로 개선되는지 확인.

In [7]:
# =========================================================
# Model 3: Baseline + 규칙형 변수 8개
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + Rule-based"
FIXED_THRESHOLD = 0.990239

model3_features = [
    # Baseline
    "category",
    "amt",
    "trans_hour",
    "age",

    # 규칙형 변수
    "is_online",
    "is_high_amt",
    "risk_time_22_04",
    "high_speed"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model3_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model3_features))
print("사용 변수:", model3_features)


# 2. 설명변수와 목표변수 생성
X_model3 = df[model3_features].copy()
y_model3 = df["is_fraud"].astype("int8").copy()

# 범주형 변수 지정
X_model3["category"] = (
    X_model3["category"]
    .astype("category")
)

print("\n변수 자료형")
print(X_model3.dtypes)

print("\n결측치 수")
print(X_model3.isna().sum())


# 3. 시간순 70:30 분할
split_index_model3 = int(len(df) * 0.70)

X_train_model3 = (
    X_model3
    .iloc[:split_index_model3]
    .copy()
)

X_valid_model3 = (
    X_model3
    .iloc[split_index_model3:]
    .copy()
)

y_train_model3 = (
    y_model3
    .iloc[:split_index_model3]
    .copy()
)

y_valid_model3 = (
    y_model3
    .iloc[split_index_model3:]
    .copy()
)

print("\nTrain:", X_train_model3.shape)
print("Valid:", X_valid_model3.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model3 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model3 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model3:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model3:,
        "trans_date_trans_time"
    ].max()
)

print("\nTrain 이상거래 건수:", int(y_train_model3.sum()))
print("Valid 이상거래 건수:", int(y_valid_model3.sum()))

print("Train 이상거래율:", y_train_model3.mean())
print("Valid 이상거래율:", y_valid_model3.mean())


# 4. 클래스 불균형 가중치 계산
negative_count_model3 = int(
    (y_train_model3 == 0).sum()
)

positive_count_model3 = int(
    (y_train_model3 == 1).sum()
)

scale_pos_weight_model3 = (
    negative_count_model3
    / positive_count_model3
)

print("\n정상거래 수:", negative_count_model3)
print("이상거래 수:", positive_count_model3)
print("scale_pos_weight:", scale_pos_weight_model3)


# 5. 동일 하이퍼파라미터로 모델 생성
model3_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model3
)


# 6. 학습
model3_7030.fit(
    X_train_model3,
    y_train_model3,

    eval_X=X_valid_model3,
    eval_y=y_valid_model3,

    eval_metric="average_precision",

    categorical_feature=[
        "category"
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model3 = model3_7030.predict_proba(
    X_valid_model3,
    num_iteration=model3_7030.best_iteration_
)[:, 1]


# 8. Baseline에서 정한 고정 임계값 적용
valid_pred_model3 = (
    valid_prob_model3 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model3 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model3_features),
    "best_iteration": model3_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model3,
        valid_prob_model3
    ),

    "roc_auc": roc_auc_score(
        y_valid_model3,
        valid_prob_model3
    ),

    "precision": precision_score(
        y_valid_model3,
        valid_pred_model3,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model3,
        valid_pred_model3,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model3,
        valid_pred_model3,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 3: "
    "Baseline + Rule-based 70:30 =========="
)

print("변수 수         :", result_model3["feature_count"])
print("Best iteration :", result_model3["best_iteration"])
print("고정 임계값     :", round(result_model3["threshold"], 6))
print("PR-AUC         :", round(result_model3["pr_auc"], 6))
print("ROC-AUC        :", round(result_model3["roc_auc"], 6))
print("Precision      :", round(result_model3["precision"], 6))
print("Recall         :", round(result_model3["recall"], 6))
print("F1-score       :", round(result_model3["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model3,
        valid_pred_model3
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model3["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model3["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model3["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model3["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model3["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 8
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'is_online', 'is_high_amt', 'risk_time_22_04', 'high_speed']

변수 자료형
category           category
amt                 float64
trans_hour            int64
age                   int64
is_online             int64
is_high_amt           int64
risk_time_22_04       int64
high_speed            int64
dtype: object

결측치 수
category           0
amt                0
trans_hour         0
age                0
is_online          0
is_high_amt        0
risk_time_22_04    0
high_speed         0
dtype: int64

Train: (907672, 8)
Valid: (389003, 8)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.0061310581152330445

정상거래 수: 902551
이상거래 수: 5121
scale_pos_weight: 176.24506932239797
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.866688	valid_0's binary_loglo

Model 3은 

PR-AUC와 Precision은 소폭 개선됐지만 

Recall과 F1이 하락했으므로, 

Baseline보다 확실히 우수하다고 보기 어렵다.

* 다만 Model 3을 바로 버리지는 x.

 PR-AUC가 더 높기 때문에 나중에 임계값을 다시 조정하면 Baseline보다 좋은 균형점이 나올 가능성 있음.  그래서 후보로는 남겨둘 수 있음.


# Model 4. Baseline + 개인별 금액 패턴 (설명변수 5개)

- amt_to_prior_median_ratio: 현재 금액이 평소 금액의 몇 배인지

이렇게 구성한 이유: 

amt_to어쩌구는 연속형 값이고 나머지 금액 패턴 관련 변수들보다 정보가 가장 풍부함.

has_prior어쩌구를 넣을지 말지 봤는데, NaN이 그대로 남아있어서 lightGBM이 알아서 처리하므로, 별도의 flag 변수 추가하면 같은 정보를 중복 제공하게 되는 것. 따라서 깔끔하게 변수 하나만 추가하는 걸로.

In [8]:
## amt_to어쩌구에 결측치 있는지 확인.

# amt_to_prior_median_ratio 결측치 확인

col = "amt_to_prior_median_ratio"

print("전체 행 수:", len(df))
print("NaN 개수:", df[col].isna().sum())
print("NaN 비율:", df[col].isna().mean())

print("\nNaN 행 일부")
print(
    df.loc[
        df[col].isna(),
        [
            "trans_date_trans_time",
            "amt",
            "prior_normal_median_amt",
            "amt_to_prior_median_ratio",
            "has_prior_normal_transaction"
        ]
    ].head(10)
)

전체 행 수: 1296675
NaN 개수: 1649
NaN 비율: 0.0012717141920681745

NaN 행 일부
  trans_date_trans_time     amt  prior_normal_median_amt  \
0   2019-01-01 00:00:18    4.97                      NaN   
1   2019-01-01 00:00:44  107.23                      NaN   
2   2019-01-01 00:00:51  220.11                      NaN   
3   2019-01-01 00:01:16   45.00                      NaN   
4   2019-01-01 00:03:06   41.96                      NaN   
5   2019-01-01 00:04:08   94.63                      NaN   
6   2019-01-01 00:04:42   44.54                      NaN   
7   2019-01-01 00:05:08   71.65                      NaN   
8   2019-01-01 00:05:18    4.27                      NaN   
9   2019-01-01 00:06:01  198.39                      NaN   

   amt_to_prior_median_ratio  has_prior_normal_transaction  
0                        NaN                             0  
1                        NaN                             0  
2                        NaN                             0  
3                        N

In [9]:
print(
    "무한대 개수:",
    np.isinf(df["amt_to_prior_median_ratio"]).sum()
)

무한대 개수: 0


In [10]:
# =========================================================
# Model 4: Baseline + 개인별 금액 비율
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + Personalized Amount Ratio"
FIXED_THRESHOLD = 0.990239

model4_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model4_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model4_features))
print("사용 변수:", model4_features)


# 2. 설명변수와 목표변수 생성
X_model4 = df[model4_features].copy()
y_model4 = df["is_fraud"].astype("int8").copy()

X_model4["category"] = (
    X_model4["category"]
    .astype("category")
)

print("\n변수 자료형")
print(X_model4.dtypes)

print("\n결측치 수")
print(X_model4.isna().sum())

print(
    "\namt_to_prior_median_ratio 기초 통계"
)
print(
    X_model4[
        "amt_to_prior_median_ratio"
    ].describe()
)


# 3. 시간순 70:30 분할
split_index_model4 = int(len(df) * 0.70)

X_train_model4 = (
    X_model4
    .iloc[:split_index_model4]
    .copy()
)

X_valid_model4 = (
    X_model4
    .iloc[split_index_model4:]
    .copy()
)

y_train_model4 = (
    y_model4
    .iloc[:split_index_model4]
    .copy()
)

y_valid_model4 = (
    y_model4
    .iloc[split_index_model4:]
    .copy()
)

print("\nTrain:", X_train_model4.shape)
print("Valid:", X_valid_model4.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model4 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model4 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model4:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model4:,
        "trans_date_trans_time"
    ].max()
)

print(
    "\nTrain 이상거래 건수:",
    int(y_train_model4.sum())
)

print(
    "Valid 이상거래 건수:",
    int(y_valid_model4.sum())
)

print(
    "Train 이상거래율:",
    y_train_model4.mean()
)

print(
    "Valid 이상거래율:",
    y_valid_model4.mean()
)


# 4. 클래스 불균형 가중치 계산
negative_count_model4 = int(
    (y_train_model4 == 0).sum()
)

positive_count_model4 = int(
    (y_train_model4 == 1).sum()
)

scale_pos_weight_model4 = (
    negative_count_model4
    / positive_count_model4
)

print(
    "\n정상거래 수:",
    negative_count_model4
)

print(
    "이상거래 수:",
    positive_count_model4
)

print(
    "scale_pos_weight:",
    scale_pos_weight_model4
)


# 5. 동일 하이퍼파라미터로 모델 생성
model4_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model4
)


# 6. 학습
model4_7030.fit(
    X_train_model4,
    y_train_model4,

    eval_X=X_valid_model4,
    eval_y=y_valid_model4,

    eval_metric="average_precision",

    categorical_feature=[
        "category"
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model4 = model4_7030.predict_proba(
    X_valid_model4,
    num_iteration=model4_7030.best_iteration_
)[:, 1]


# 8. Baseline에서 정한 고정 임계값 적용
valid_pred_model4 = (
    valid_prob_model4 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model4 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model4_features),
    "best_iteration": model4_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model4,
        valid_prob_model4
    ),

    "roc_auc": roc_auc_score(
        y_valid_model4,
        valid_prob_model4
    ),

    "precision": precision_score(
        y_valid_model4,
        valid_pred_model4,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model4,
        valid_pred_model4,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model4,
        valid_pred_model4,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 4: "
    "Baseline + Personalized Amount Ratio 70:30 =========="
)

print(
    "변수 수         :",
    result_model4["feature_count"]
)

print(
    "Best iteration :",
    result_model4["best_iteration"]
)

print(
    "고정 임계값     :",
    round(
        result_model4["threshold"],
        6
    )
)

print(
    "PR-AUC         :",
    round(
        result_model4["pr_auc"],
        6
    )
)

print(
    "ROC-AUC        :",
    round(
        result_model4["roc_auc"],
        6
    )
)

print(
    "Precision      :",
    round(
        result_model4["precision"],
        6
    )
)

print(
    "Recall         :",
    round(
        result_model4["recall"],
        6
    )
)

print(
    "F1-score       :",
    round(
        result_model4["f1"],
        6
    )
)

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model4,
        valid_pred_model4
    )
)


# 11. Baseline 대비 변화
print(
    "\n========== Baseline 대비 변화 =========="
)

print(
    "PR-AUC 변화:",
    round(
        result_model4["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model4["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model4["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model4["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model4["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 5
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'amt_to_prior_median_ratio']

변수 자료형
category                     category
amt                           float64
trans_hour                      int64
age                             int64
amt_to_prior_median_ratio     float64
dtype: object

결측치 수
category                        0
amt                             0
trans_hour                      0
age                             0
amt_to_prior_median_ratio    1649
dtype: int64

amt_to_prior_median_ratio 기초 통계
count    1.295026e+06
mean     1.643860e+00
std      4.558937e+00
min      2.176430e-03
25%      2.606922e-01
50%      1.003543e+00
75%      1.894498e+00
max      8.557099e+02
Name: amt_to_prior_median_ratio, dtype: float64

Train: (907672, 5)
Valid: (389003, 5)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.0061310581152330445

Model4가 현재까지 가장 좋은 후보.

PR-AUC basline 대비 +0.016, precision +0.018, Recall -0.005

고정 임계값에서는 Recall이 약간 줄었지만 Precision 과 F1이 개선됨.

# Model 5. Baseline + 개인별 시간대 이탈

baseline + outside_trans_hours_80

In [11]:
# =========================================================
# Model 5: Baseline + 개인별 거래시간 이탈
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + Outside Active Hours"
FIXED_THRESHOLD = 0.990239

model5_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "outside_trans_hours_80"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model5_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model5_features))
print("사용 변수:", model5_features)


# 2. 설명변수와 목표변수 생성
X_model5 = df[model5_features].copy()
y_model5 = df["is_fraud"].astype("int8").copy()

X_model5["category"] = (
    X_model5["category"]
    .astype("category")
)

print("\n변수 자료형")
print(X_model5.dtypes)

print("\n결측치 수")
print(X_model5.isna().sum())

print("\noutside_trans_hours_80 값 분포")
print(
    X_model5["outside_trans_hours_80"]
    .value_counts(dropna=False)
    .sort_index()
)


# 3. 시간순 70:30 분할
split_index_model5 = int(len(df) * 0.70)

X_train_model5 = (
    X_model5
    .iloc[:split_index_model5]
    .copy()
)

X_valid_model5 = (
    X_model5
    .iloc[split_index_model5:]
    .copy()
)

y_train_model5 = (
    y_model5
    .iloc[:split_index_model5]
    .copy()
)

y_valid_model5 = (
    y_model5
    .iloc[split_index_model5:]
    .copy()
)

print("\nTrain:", X_train_model5.shape)
print("Valid:", X_valid_model5.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model5 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model5 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model5:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model5:,
        "trans_date_trans_time"
    ].max()
)

print(
    "\nTrain 이상거래 건수:",
    int(y_train_model5.sum())
)

print(
    "Valid 이상거래 건수:",
    int(y_valid_model5.sum())
)

print(
    "Train 이상거래율:",
    y_train_model5.mean()
)

print(
    "Valid 이상거래율:",
    y_valid_model5.mean()
)


# 4. 클래스 불균형 가중치 계산
negative_count_model5 = int(
    (y_train_model5 == 0).sum()
)

positive_count_model5 = int(
    (y_train_model5 == 1).sum()
)

scale_pos_weight_model5 = (
    negative_count_model5
    / positive_count_model5
)

print("\n정상거래 수:", negative_count_model5)
print("이상거래 수:", positive_count_model5)
print("scale_pos_weight:", scale_pos_weight_model5)


# 5. Baseline과 동일한 하이퍼파라미터로 모델 생성
model5_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model5
)


# 6. 학습
model5_7030.fit(
    X_train_model5,
    y_train_model5,

    eval_X=X_valid_model5,
    eval_y=y_valid_model5,

    eval_metric="average_precision",

    categorical_feature=[
        "category"
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model5 = model5_7030.predict_proba(
    X_valid_model5,
    num_iteration=model5_7030.best_iteration_
)[:, 1]


# 8. Baseline에서 정한 고정 임계값 적용
valid_pred_model5 = (
    valid_prob_model5 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model5 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model5_features),
    "best_iteration": model5_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model5,
        valid_prob_model5
    ),

    "roc_auc": roc_auc_score(
        y_valid_model5,
        valid_prob_model5
    ),

    "precision": precision_score(
        y_valid_model5,
        valid_pred_model5,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model5,
        valid_pred_model5,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model5,
        valid_pred_model5,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 5: "
    "Baseline + Outside Active Hours 70:30 =========="
)

print("변수 수         :", result_model5["feature_count"])
print("Best iteration :", result_model5["best_iteration"])
print("고정 임계값     :", round(result_model5["threshold"], 6))
print("PR-AUC         :", round(result_model5["pr_auc"], 6))
print("ROC-AUC        :", round(result_model5["roc_auc"], 6))
print("Precision      :", round(result_model5["precision"], 6))
print("Recall         :", round(result_model5["recall"], 6))
print("F1-score       :", round(result_model5["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model5,
        valid_pred_model5
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model5["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model5["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model5["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model5["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model5["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 5
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'outside_trans_hours_80']

변수 자료형
category                  category
amt                        float64
trans_hour                   int64
age                          int64
outside_trans_hours_80       int64
dtype: object

결측치 수
category                  0
amt                       0
trans_hour                0
age                       0
outside_trans_hours_80    0
dtype: int64

outside_trans_hours_80 값 분포
outside_trans_hours_80
0    1028541
1     268134
Name: count, dtype: int64

Train: (907672, 5)
Valid: (389003, 5)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.0061310581152330445

정상거래 수: 902551
이상거래 수: 5121
scale_pos_weight: 176.24506932239797
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.857803	valid_0's binary_logloss: 0.05

Model 5는 baseline 대비 성능 개선 거의 없음. 

단, outside_trans_hours_80은 정상거래를 더 보수적으로 거르는데는 조금 도움이 됨. 하지만 이상거래는 더 많이 놓침. 단독 추가 변수로서의 효과는 제한적.

# Model 6. Baseline + count_30min

In [12]:
# =========================================================
# Model 6: Baseline + 최근 30분 거래 횟수
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + 30min Transaction Count"
FIXED_THRESHOLD = 0.990239

model6_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "count_30min"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model6_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model6_features))
print("사용 변수:", model6_features)


# 2. 설명변수와 목표변수 생성
X_model6 = df[model6_features].copy()
y_model6 = df["is_fraud"].astype("int8").copy()

X_model6["category"] = X_model6["category"].astype("category")

print("\n변수 자료형")
print(X_model6.dtypes)

print("\n결측치 수")
print(X_model6.isna().sum())

print("\ncount_30min 기초 통계")
print(X_model6["count_30min"].describe())

print("\ncount_30min 값 분포 상위 15개")
print(
    X_model6["count_30min"]
    .value_counts(dropna=False)
    .sort_index()
    .head(15)
)


# 3. 시간순 70:30 분할
split_index_model6 = int(len(df) * 0.70)

X_train_model6 = X_model6.iloc[:split_index_model6].copy()
X_valid_model6 = X_model6.iloc[split_index_model6:].copy()

y_train_model6 = y_model6.iloc[:split_index_model6].copy()
y_valid_model6 = y_model6.iloc[split_index_model6:].copy()

print("\nTrain:", X_train_model6.shape)
print("Valid:", X_valid_model6.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model6 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model6 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model6:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model6:,
        "trans_date_trans_time"
    ].max()
)

print("\nTrain 이상거래 건수:", int(y_train_model6.sum()))
print("Valid 이상거래 건수:", int(y_valid_model6.sum()))

print("Train 이상거래율:", y_train_model6.mean())
print("Valid 이상거래율:", y_valid_model6.mean())


# 4. 클래스 불균형 가중치 계산
negative_count_model6 = int((y_train_model6 == 0).sum())
positive_count_model6 = int((y_train_model6 == 1).sum())

scale_pos_weight_model6 = (
    negative_count_model6 / positive_count_model6
)

print("\n정상거래 수:", negative_count_model6)
print("이상거래 수:", positive_count_model6)
print("scale_pos_weight:", scale_pos_weight_model6)


# 5. 동일 하이퍼파라미터로 모델 생성
model6_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model6
)


# 6. 학습
model6_7030.fit(
    X_train_model6,
    y_train_model6,

    eval_X=X_valid_model6,
    eval_y=y_valid_model6,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model6 = model6_7030.predict_proba(
    X_valid_model6,
    num_iteration=model6_7030.best_iteration_
)[:, 1]


# 8. Baseline에서 정한 고정 임계값 적용
valid_pred_model6 = (
    valid_prob_model6 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model6 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model6_features),
    "best_iteration": model6_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model6,
        valid_prob_model6
    ),

    "roc_auc": roc_auc_score(
        y_valid_model6,
        valid_prob_model6
    ),

    "precision": precision_score(
        y_valid_model6,
        valid_pred_model6,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model6,
        valid_pred_model6,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model6,
        valid_pred_model6,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 6: "
    "Baseline + 30min Transaction Count 70:30 =========="
)

print("변수 수         :", result_model6["feature_count"])
print("Best iteration :", result_model6["best_iteration"])
print("고정 임계값     :", round(result_model6["threshold"], 6))
print("PR-AUC         :", round(result_model6["pr_auc"], 6))
print("ROC-AUC        :", round(result_model6["roc_auc"], 6))
print("Precision      :", round(result_model6["precision"], 6))
print("Recall         :", round(result_model6["recall"], 6))
print("F1-score       :", round(result_model6["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model6,
        valid_pred_model6
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model6["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model6["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model6["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model6["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model6["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 5
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'count_30min']

변수 자료형
category       category
amt             float64
trans_hour        int64
age               int64
count_30min       int64
dtype: object

결측치 수
category       0
amt            0
trans_hour     0
age            0
count_30min    0
dtype: int64

count_30min 기초 통계
count    1.296675e+06
mean     9.095595e-01
std      4.783908e-01
min      0.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      5.000000e+00
Name: count_30min, dtype: float64

count_30min 값 분포 상위 15개
count_30min
0     206282
1    1007030
2      78081
3       4939
4        321
5         22
Name: count, dtype: int64

Train: (907672, 5)
Valid: (389003, 5)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.0061310581152330445

정상거래 수: 902551
이상거래 수: 5121
scale_pos_weight: 176.2450

Model 6는 5보단 낫지만 4보다는 약함.

count_30min은 PR-AUC는 어느 정도 높였지만, 

고정 임계값에서는 이상거래를 더 놓쳐서(24건 증가) Recall 과 F1이 감소함.

# Model 7. baseline + rolling_sum_amt_1h

In [13]:
# =========================================================
# Model 7: Baseline + 최근 1시간 누적 거래금액
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + 1h Rolling Amount Sum"
FIXED_THRESHOLD = 0.990239

model7_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "rolling_sum_amt_1h"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model7_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model7_features))
print("사용 변수:", model7_features)


# 2. 설명변수와 목표변수 생성
X_model7 = df[model7_features].copy()
y_model7 = df["is_fraud"].astype("int8").copy()

X_model7["category"] = X_model7["category"].astype("category")

print("\n변수 자료형")
print(X_model7.dtypes)

print("\n결측치 수")
print(X_model7.isna().sum())

print("\nrolling_sum_amt_1h 기초 통계")
print(X_model7["rolling_sum_amt_1h"].describe())

print("\n무한대 개수")
print(
    np.isinf(
        X_model7["rolling_sum_amt_1h"]
    ).sum()
)


# 3. 시간순 70:30 분할
split_index_model7 = int(len(df) * 0.70)

X_train_model7 = X_model7.iloc[:split_index_model7].copy()
X_valid_model7 = X_model7.iloc[split_index_model7:].copy()

y_train_model7 = y_model7.iloc[:split_index_model7].copy()
y_valid_model7 = y_model7.iloc[split_index_model7:].copy()

print("\nTrain:", X_train_model7.shape)
print("Valid:", X_valid_model7.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model7 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model7 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model7:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model7:,
        "trans_date_trans_time"
    ].max()
)

print("\nTrain 이상거래 건수:", int(y_train_model7.sum()))
print("Valid 이상거래 건수:", int(y_valid_model7.sum()))

print("Train 이상거래율:", y_train_model7.mean())
print("Valid 이상거래율:", y_valid_model7.mean())


# 4. 클래스 불균형 가중치 계산
negative_count_model7 = int(
    (y_train_model7 == 0).sum()
)

positive_count_model7 = int(
    (y_train_model7 == 1).sum()
)

scale_pos_weight_model7 = (
    negative_count_model7
    / positive_count_model7
)

print("\n정상거래 수:", negative_count_model7)
print("이상거래 수:", positive_count_model7)
print("scale_pos_weight:", scale_pos_weight_model7)


# 5. 동일 하이퍼파라미터로 모델 생성
model7_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model7
)


# 6. 학습
model7_7030.fit(
    X_train_model7,
    y_train_model7,

    eval_X=X_valid_model7,
    eval_y=y_valid_model7,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model7 = model7_7030.predict_proba(
    X_valid_model7,
    num_iteration=model7_7030.best_iteration_
)[:, 1]


# 8. Baseline 고정 임계값 적용
valid_pred_model7 = (
    valid_prob_model7 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model7 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model7_features),
    "best_iteration": model7_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model7,
        valid_prob_model7
    ),

    "roc_auc": roc_auc_score(
        y_valid_model7,
        valid_prob_model7
    ),

    "precision": precision_score(
        y_valid_model7,
        valid_pred_model7,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model7,
        valid_pred_model7,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model7,
        valid_pred_model7,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 7: "
    "Baseline + 1h Rolling Amount Sum 70:30 =========="
)

print("변수 수         :", result_model7["feature_count"])
print("Best iteration :", result_model7["best_iteration"])
print("고정 임계값     :", round(result_model7["threshold"], 6))
print("PR-AUC         :", round(result_model7["pr_auc"], 6))
print("ROC-AUC        :", round(result_model7["roc_auc"], 6))
print("Precision      :", round(result_model7["precision"], 6))
print("Recall         :", round(result_model7["recall"], 6))
print("F1-score       :", round(result_model7["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model7,
        valid_pred_model7
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model7["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model7["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model7["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model7["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model7["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 5
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'rolling_sum_amt_1h']

변수 자료형
category              category
amt                    float64
trans_hour               int64
age                      int64
rolling_sum_amt_1h     float64
dtype: object

결측치 수
category              0
amt                   0
trans_hour            0
age                   0
rolling_sum_amt_1h    0
dtype: int64

rolling_sum_amt_1h 기초 통계
count    1.296675e+06
mean     8.491117e+01
std      1.947081e+02
min      1.000000e+00
25%      1.425000e+01
50%      5.429000e+01
75%      9.630000e+01
max      2.894890e+04
Name: rolling_sum_amt_1h, dtype: float64

무한대 개수
0

Train: (907672, 5)
Valid: (389003, 5)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.0061310581152330445

정상거래 수: 902551
이상거래 수: 5121
scale_pos_weight: 176.24506932239797
Training until validation sco

Model 7은 현재까지 가장 좋은 모델.
Precision과 Recall 둘 다 개선됨.

오탐 44건 감소, 미탐 48건 감소, 탐지 성공 48건 증가.

# Model 8. baseline + recent_24h_high_amt_count

In [14]:
# =========================================================
# Model 8: Baseline + 최근 24시간 고액거래 횟수
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + Recent 24h High Amount Count"
FIXED_THRESHOLD = 0.990239

model8_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model8_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model8_features))
print("사용 변수:", model8_features)


# 2. 설명변수와 목표변수 생성
X_model8 = df[model8_features].copy()
y_model8 = df["is_fraud"].astype("int8").copy()

X_model8["category"] = X_model8["category"].astype("category")

print("\n변수 자료형")
print(X_model8.dtypes)

print("\n결측치 수")
print(X_model8.isna().sum())

print("\nrecent_24h_high_amt_count 기초 통계")
print(
    X_model8["recent_24h_high_amt_count"].describe()
)

print("\nrecent_24h_high_amt_count 값 분포")
print(
    X_model8["recent_24h_high_amt_count"]
    .value_counts(dropna=False)
    .sort_index()
    .head(20)
)


# 3. 시간순 70:30 분할
split_index_model8 = int(len(df) * 0.70)

X_train_model8 = X_model8.iloc[:split_index_model8].copy()
X_valid_model8 = X_model8.iloc[split_index_model8:].copy()

y_train_model8 = y_model8.iloc[:split_index_model8].copy()
y_valid_model8 = y_model8.iloc[split_index_model8:].copy()

print("\nTrain:", X_train_model8.shape)
print("Valid:", X_valid_model8.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model8 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model8 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model8:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model8:,
        "trans_date_trans_time"
    ].max()
)

print("\nTrain 이상거래 건수:", int(y_train_model8.sum()))
print("Valid 이상거래 건수:", int(y_valid_model8.sum()))

print("Train 이상거래율:", y_train_model8.mean())
print("Valid 이상거래율:", y_valid_model8.mean())


# 4. 클래스 불균형 가중치 계산
negative_count_model8 = int(
    (y_train_model8 == 0).sum()
)

positive_count_model8 = int(
    (y_train_model8 == 1).sum()
)

scale_pos_weight_model8 = (
    negative_count_model8
    / positive_count_model8
)

print("\n정상거래 수:", negative_count_model8)
print("이상거래 수:", positive_count_model8)
print("scale_pos_weight:", scale_pos_weight_model8)


# 5. 동일 하이퍼파라미터로 모델 생성
model8_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model8
)


# 6. 학습
model8_7030.fit(
    X_train_model8,
    y_train_model8,

    eval_X=X_valid_model8,
    eval_y=y_valid_model8,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model8 = model8_7030.predict_proba(
    X_valid_model8,
    num_iteration=model8_7030.best_iteration_
)[:, 1]


# 8. Baseline 고정 임계값 적용
valid_pred_model8 = (
    valid_prob_model8 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model8 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model8_features),
    "best_iteration": model8_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model8,
        valid_prob_model8
    ),

    "roc_auc": roc_auc_score(
        y_valid_model8,
        valid_prob_model8
    ),

    "precision": precision_score(
        y_valid_model8,
        valid_pred_model8,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model8,
        valid_pred_model8,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model8,
        valid_pred_model8,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 8: "
    "Baseline + Recent 24h High Amount Count 70:30 =========="
)

print("변수 수         :", result_model8["feature_count"])
print("Best iteration :", result_model8["best_iteration"])
print("고정 임계값     :", round(result_model8["threshold"], 6))
print("PR-AUC         :", round(result_model8["pr_auc"], 6))
print("ROC-AUC        :", round(result_model8["roc_auc"], 6))
print("Precision      :", round(result_model8["precision"], 6))
print("Recall         :", round(result_model8["recall"], 6))
print("F1-score       :", round(result_model8["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model8,
        valid_pred_model8
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model8["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model8["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model8["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model8["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model8["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 5
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'recent_24h_high_amt_count']

변수 자료형
category                     category
amt                           float64
trans_hour                      int64
age                             int64
recent_24h_high_amt_count       int64
dtype: object

결측치 수
category                     0
amt                          0
trans_hour                   0
age                          0
recent_24h_high_amt_count    0
dtype: int64

recent_24h_high_amt_count 기초 통계
count    1.296675e+06
mean     4.816936e-02
std      2.741523e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      9.000000e+00
Name: recent_24h_high_amt_count, dtype: float64

recent_24h_high_amt_count 값 분포
recent_24h_high_amt_count
0    1244934
1      45913
2       3132
3       1342
4        792
5        348
6        161
7         42
8         10
9          1
Name: count, dtype: int64

Train: (907672, 5)
Valid: (389003, 5)
Train 기간: 2019-

Model 8은 압도적으로 가장 좋음. 다만 성능 상승 폭이 매우 커서, 현재 거래나 미래 거래를 포함하지 않았는지 확인해야 함.

- 오탐 226 -> 98건
- 미탐 483 -> 267건
- 탐지 성공 : 1902 -> 2118건

In [15]:
# =========================================================
# recent_24h_high_amt_count 현재 거래 포함 여부 1차 점검
# =========================================================

check_df = (
    df.sort_values(
        ["cc_num", "trans_date_trans_time"]
    )
    .copy()
)

# 고객별 데이터상 최초 거래
first_transactions = (
    check_df
    .groupby("cc_num", observed=True)
    .head(1)
)

print("고객별 최초 거래 수:", len(first_transactions))

print("\n최초 거래의 recent_24h_high_amt_count 분포")
print(
    first_transactions[
        "recent_24h_high_amt_count"
    ]
    .value_counts(dropna=False)
    .sort_index()
)

print("\n최초 거래인데 count가 1 이상인 행 수")
print(
    (
        first_transactions[
            "recent_24h_high_amt_count"
        ] >= 1
    ).sum()
)

print("\n최초 거래 일부")
print(
    first_transactions[
        [
            "cc_num",
            "trans_date_trans_time",
            "amt",
            "recent_24h_high_amt_count"
        ]
    ].head(20)
)

고객별 최초 거래 수: 983

최초 거래의 recent_24h_high_amt_count 분포
recent_24h_high_amt_count
0    983
Name: count, dtype: int64

최초 거래인데 count가 1 이상인 행 수
0

최초 거래 일부
               cc_num trans_date_trans_time     amt  recent_24h_high_amt_count
1017      60416207185   2019-01-01 12:47:15    7.27                          0
4415      60422928733   2019-01-03 18:38:26   94.20                          0
516       60423098130   2019-01-01 06:48:36    5.68                          0
586       60427851591   2019-01-01 07:36:27   78.80                          0
7892      60487002085   2019-01-06 03:23:55    9.37                          0
984       60490596305   2019-01-01 12:31:09   87.07                          0
53        60495593109   2019-01-01 00:39:43  122.86                          0
252      501802953619   2019-01-01 03:12:06    8.68                          0
1150401  501818133297   2020-04-24 22:09:30  977.70                          0
525      501828204849   2019-01-01 06:53:54    7.12      

다만 이 검사는 1차 확인. 

완전하게 보려면 임의의 고객 몇 명에 대해 실제 과거 24시간 고액거래 횟수와 변수값이 일치하는지 직접 재계산하는 검사가 가장 확실함. 

그래도 지금 단계에서는 Model 8을 유효 후보로 유지.

# Model 9. baseline + speed_2

In [16]:
# =========================================================
# Model 9: Baseline + 이전 거래 대비 이동속도
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + Transaction Speed"
FIXED_THRESHOLD = 0.990239

model9_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "speed_2"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model9_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model9_features))
print("사용 변수:", model9_features)


# 2. 설명변수와 목표변수 생성
X_model9 = df[model9_features].copy()
y_model9 = df["is_fraud"].astype("int8").copy()

X_model9["category"] = X_model9["category"].astype("category")

print("\n변수 자료형")
print(X_model9.dtypes)

print("\n결측치 수")
print(X_model9.isna().sum())

print("\nspeed_2 기초 통계")
print(X_model9["speed_2"].describe())

print("\nspeed_2 무한대 개수")
print(
    np.isinf(
        X_model9["speed_2"]
    ).sum()
)

print("\nspeed_2 상위 분위수")
print(
    X_model9["speed_2"].quantile(
        [0.90, 0.95, 0.99, 0.999, 1.0]
    )
)


# 3. 시간순 70:30 분할
split_index_model9 = int(len(df) * 0.70)

X_train_model9 = X_model9.iloc[:split_index_model9].copy()
X_valid_model9 = X_model9.iloc[split_index_model9:].copy()

y_train_model9 = y_model9.iloc[:split_index_model9].copy()
y_valid_model9 = y_model9.iloc[split_index_model9:].copy()

print("\nTrain:", X_train_model9.shape)
print("Valid:", X_valid_model9.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model9 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model9 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model9:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model9:,
        "trans_date_trans_time"
    ].max()
)

print("\nTrain 이상거래 건수:", int(y_train_model9.sum()))
print("Valid 이상거래 건수:", int(y_valid_model9.sum()))

print("Train 이상거래율:", y_train_model9.mean())
print("Valid 이상거래율:", y_valid_model9.mean())


# 4. 클래스 불균형 가중치 계산
negative_count_model9 = int(
    (y_train_model9 == 0).sum()
)

positive_count_model9 = int(
    (y_train_model9 == 1).sum()
)

scale_pos_weight_model9 = (
    negative_count_model9
    / positive_count_model9
)

print("\n정상거래 수:", negative_count_model9)
print("이상거래 수:", positive_count_model9)
print("scale_pos_weight:", scale_pos_weight_model9)


# 5. 동일 하이퍼파라미터로 모델 생성
model9_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model9
)


# 6. 학습
model9_7030.fit(
    X_train_model9,
    y_train_model9,

    eval_X=X_valid_model9,
    eval_y=y_valid_model9,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model9 = model9_7030.predict_proba(
    X_valid_model9,
    num_iteration=model9_7030.best_iteration_
)[:, 1]


# 8. Baseline 고정 임계값 적용
valid_pred_model9 = (
    valid_prob_model9 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model9 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model9_features),
    "best_iteration": model9_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model9,
        valid_prob_model9
    ),

    "roc_auc": roc_auc_score(
        y_valid_model9,
        valid_prob_model9
    ),

    "precision": precision_score(
        y_valid_model9,
        valid_pred_model9,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model9,
        valid_pred_model9,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model9,
        valid_pred_model9,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 9: "
    "Baseline + Transaction Speed 70:30 =========="
)

print("변수 수         :", result_model9["feature_count"])
print("Best iteration :", result_model9["best_iteration"])
print("고정 임계값     :", round(result_model9["threshold"], 6))
print("PR-AUC         :", round(result_model9["pr_auc"], 6))
print("ROC-AUC        :", round(result_model9["roc_auc"], 6))
print("Precision      :", round(result_model9["precision"], 6))
print("Recall         :", round(result_model9["recall"], 6))
print("F1-score       :", round(result_model9["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model9,
        valid_pred_model9
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model9["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model9["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model9["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model9["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model9["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 5
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'speed_2']

변수 자료형
category      category
amt            float64
trans_hour       int64
age              int64
speed_2        float64
dtype: object

결측치 수
category      0
amt           0
trans_hour    0
age           0
speed_2       0
dtype: int64

speed_2 기초 통계
count    1.296675e+06
mean     8.605355e+01
std      4.590036e+02
min      0.000000e+00
25%      2.968340e+00
50%      1.162299e+01
75%      3.933591e+01
max      1.582018e+04
Name: speed_2, dtype: float64

speed_2 무한대 개수
0

speed_2 상위 분위수
0.900      124.781088
0.950      267.436410
0.990     1403.208303
0.999     7505.267218
1.000    15820.182333
Name: speed_2, dtype: float64

Train: (907672, 5)
Valid: (389003, 5)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.0061310581152330445

정상거래 수: 902551
이상거래 수: 5121
scale_pos_weigh

Model 9(speed_2)는 PR-AUC를 조금 개선했지만, 최종 후보로 단독채택할정도는 x

In [17]:
# =========================================================
# category_recent_fraud_rate 1차 누수 점검
# 각 업종의 데이터상 최초 거래 확인
# =========================================================

check_category = (
    df.sort_values("trans_date_trans_time")
    .copy()
)

first_category_transactions = (
    check_category
    .groupby("category", observed=True)
    .head(1)
)

print("업종 수:", len(first_category_transactions))

print("\n업종별 최초 거래의 최근 7일 이상거래율")
print(
    first_category_transactions[
        [
            "category",
            "trans_date_trans_time",
            "is_fraud",
            "category_recent_fraud_rate"
        ]
    ]
    .sort_values("category")
    .to_string(index=False)
)

print("\n최초 거래 변수값 분포")
print(
    first_category_transactions[
        "category_recent_fraud_rate"
    ].value_counts(dropna=False)
)

print("\n결측치 수")
print(
    df["category_recent_fraud_rate"].isna().sum()
)

print("\n기초 통계")
print(
    df["category_recent_fraud_rate"].describe()
)

업종 수: 14

업종별 최초 거래의 최근 7일 이상거래율
      category trans_date_trans_time  is_fraud  category_recent_fraud_rate
 entertainment   2019-01-01 00:00:51         0                         0.0
   food_dining   2019-01-01 00:11:14         0                         0.0
 gas_transport   2019-01-01 00:01:16         0                         0.0
   grocery_net   2019-01-01 00:04:42         0                         0.0
   grocery_pos   2019-01-01 00:00:44         0                         0.0
health_fitness   2019-01-01 12:00:35         0                         0.0
          home   2019-01-01 12:18:39         0                         0.0
     kids_pets   2019-01-01 12:04:54         0                         0.0
      misc_net   2019-01-01 00:00:18         0                         0.0
      misc_pos   2019-01-01 00:03:06         0                         0.0
 personal_care   2019-01-01 12:00:31         0                         0.0
  shopping_net   2019-01-01 00:06:53         0                     

업종별 최초 거래 14건의 is_fraud가 전부 0이기 때문에 데이터 누수 문제가 발생하지 않았다고 할 수 없음. 현재 거래가 잘못 포함됐더라도 최초 거래의 계산값이 0으로 나올 수 있어서, 지금 결과만으로는 구분이 안 됨.

Model 10을 돌리기 전에 category_recent_fraud_rate를 현재 거래를 제외한 과거 7일 데이터로 직접 다시 계산한 값과 비교해야 함. 아래 코드.

In [18]:
# =========================================================
# category_recent_fraud_rate 정밀 누수 점검
# 현재 거래를 제외한 과거 7일 이상거래율 직접 재계산
# =========================================================

check_df = (
    df[
        [
            "category",
            "trans_date_trans_time",
            "is_fraud",
            "category_recent_fraud_rate"
        ]
    ]
    .sort_values("trans_date_trans_time")
    .copy()
)

check_df["trans_date_trans_time"] = pd.to_datetime(
    check_df["trans_date_trans_time"]
)

# 원래 행 위치 보존
check_df["_original_index"] = check_df.index

recalculated_parts = []

for category_name, group in check_df.groupby(
    "category",
    observed=True,
    sort=False
):
    group = (
        group
        .sort_values("trans_date_trans_time")
        .copy()
    )

    group = group.set_index("trans_date_trans_time")

    # 현재 거래를 제외하기 위해 closed="left" 사용
    past_fraud_sum = (
        group["is_fraud"]
        .rolling("7D", closed="left")
        .sum()
    )

    past_transaction_count = (
        group["is_fraud"]
        .rolling("7D", closed="left")
        .count()
    )

    group["recalculated_category_fraud_rate"] = (
        past_fraud_sum
        / past_transaction_count
    ).fillna(0)

    recalculated_parts.append(
        group.reset_index()
    )

recalculated_df = pd.concat(
    recalculated_parts,
    ignore_index=True
)

recalculated_df = (
    recalculated_df
    .set_index("_original_index")
    .sort_index()
)

# 기존 변수와 직접 재계산값의 차이
recalculated_df["rate_difference"] = (
    recalculated_df["category_recent_fraud_rate"]
    - recalculated_df["recalculated_category_fraud_rate"]
).abs()

print("전체 행 수:", len(recalculated_df))

print(
    "\n완전히 일치하는 행 수:",
    np.isclose(
        recalculated_df["category_recent_fraud_rate"],
        recalculated_df["recalculated_category_fraud_rate"],
        atol=1e-12
    ).sum()
)

print(
    "불일치 행 수:",
    (
        ~np.isclose(
            recalculated_df["category_recent_fraud_rate"],
            recalculated_df["recalculated_category_fraud_rate"],
            atol=1e-12
        )
    ).sum()
)

print(
    "\n최대 절대 차이:",
    recalculated_df["rate_difference"].max()
)

print(
    "평균 절대 차이:",
    recalculated_df["rate_difference"].mean()
)

print("\n차이가 큰 행 일부")
print(
    recalculated_df.loc[
        recalculated_df["rate_difference"] > 1e-12,
        [
            "trans_date_trans_time",
            "category",
            "is_fraud",
            "category_recent_fraud_rate",
            "recalculated_category_fraud_rate",
            "rate_difference"
        ]
    ]
    .sort_values(
        "rate_difference",
        ascending=False
    )
    .head(20)
)

전체 행 수: 1296675

완전히 일치하는 행 수: 1296675
불일치 행 수: 0

최대 절대 차이: 9.996344030316351e-17
평균 절대 차이: 4.308784947421991e-17

차이가 큰 행 일부
Empty DataFrame
Columns: [trans_date_trans_time, category, is_fraud, category_recent_fraud_rate, recalculated_category_fraud_rate, rate_difference]
Index: []


ㅇㅋ 데이터 누수 없는 거 확인.

In [19]:
# =========================================================
# Model 10: Baseline + 업종별 최근 7일 이상거래율
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + Category Recent Fraud Rate"
FIXED_THRESHOLD = 0.990239

model10_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "category_recent_fraud_rate"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model10_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model10_features))
print("사용 변수:", model10_features)


# 2. 설명변수와 목표변수 생성
X_model10 = df[model10_features].copy()
y_model10 = df["is_fraud"].astype("int8").copy()

X_model10["category"] = (
    X_model10["category"]
    .astype("category")
)

print("\n변수 자료형")
print(X_model10.dtypes)

print("\n결측치 수")
print(X_model10.isna().sum())

print("\ncategory_recent_fraud_rate 기초 통계")
print(
    X_model10[
        "category_recent_fraud_rate"
    ].describe()
)

print("\n무한대 개수")
print(
    np.isinf(
        X_model10["category_recent_fraud_rate"]
    ).sum()
)


# 3. 시간순 70:30 분할
split_index_model10 = int(len(df) * 0.70)

X_train_model10 = (
    X_model10
    .iloc[:split_index_model10]
    .copy()
)

X_valid_model10 = (
    X_model10
    .iloc[split_index_model10:]
    .copy()
)

y_train_model10 = (
    y_model10
    .iloc[:split_index_model10]
    .copy()
)

y_valid_model10 = (
    y_model10
    .iloc[split_index_model10:]
    .copy()
)

print("\nTrain:", X_train_model10.shape)
print("Valid:", X_valid_model10.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model10 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model10 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model10:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model10:,
        "trans_date_trans_time"
    ].max()
)

print(
    "\nTrain 이상거래 건수:",
    int(y_train_model10.sum())
)

print(
    "Valid 이상거래 건수:",
    int(y_valid_model10.sum())
)

print(
    "Train 이상거래율:",
    y_train_model10.mean()
)

print(
    "Valid 이상거래율:",
    y_valid_model10.mean()
)


# 4. 클래스 불균형 가중치 계산
negative_count_model10 = int(
    (y_train_model10 == 0).sum()
)

positive_count_model10 = int(
    (y_train_model10 == 1).sum()
)

scale_pos_weight_model10 = (
    negative_count_model10
    / positive_count_model10
)

print("\n정상거래 수:", negative_count_model10)
print("이상거래 수:", positive_count_model10)
print("scale_pos_weight:", scale_pos_weight_model10)


# 5. 동일 하이퍼파라미터로 모델 생성
model10_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model10
)


# 6. 학습
model10_7030.fit(
    X_train_model10,
    y_train_model10,

    eval_X=X_valid_model10,
    eval_y=y_valid_model10,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model10 = model10_7030.predict_proba(
    X_valid_model10,
    num_iteration=model10_7030.best_iteration_
)[:, 1]


# 8. Baseline 고정 임계값 적용
valid_pred_model10 = (
    valid_prob_model10 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model10 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model10_features),
    "best_iteration": model10_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model10,
        valid_prob_model10
    ),

    "roc_auc": roc_auc_score(
        y_valid_model10,
        valid_prob_model10
    ),

    "precision": precision_score(
        y_valid_model10,
        valid_pred_model10,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model10,
        valid_pred_model10,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model10,
        valid_pred_model10,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 10: "
    "Baseline + Category Recent Fraud Rate 70:30 =========="
)

print("변수 수         :", result_model10["feature_count"])
print("Best iteration :", result_model10["best_iteration"])
print("고정 임계값     :", round(result_model10["threshold"], 6))
print("PR-AUC         :", round(result_model10["pr_auc"], 6))
print("ROC-AUC        :", round(result_model10["roc_auc"], 6))
print("Precision      :", round(result_model10["precision"], 6))
print("Recall         :", round(result_model10["recall"], 6))
print("F1-score       :", round(result_model10["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model10,
        valid_pred_model10
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model10["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model10["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model10["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model10["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model10["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 5
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'category_recent_fraud_rate']

변수 자료형
category                      category
amt                            float64
trans_hour                       int64
age                              int64
category_recent_fraud_rate     float64
dtype: object

결측치 수
category                      0
amt                           0
trans_hour                    0
age                           0
category_recent_fraud_rate    0
dtype: int64

category_recent_fraud_rate 기초 통계
count    1.296675e+06
mean     5.763931e-03
std      6.550658e-03
min      0.000000e+00
25%      1.458789e-03
50%      3.311258e-03
75%      7.770472e-03
max      5.326460e-02
Name: category_recent_fraud_rate, dtype: float64

무한대 개수
0

Train: (907672, 5)
Valid: (389003, 5)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.00613105811

Model 10(업종별 최근 7일 내 사기율) 은 제외.

독립 효과 실험 끝. 이제 결과를 기준으로 유효한 변수 조합 모델 시작.

# Model 11. recent_24h_high_amt_count + rolling_sum_amt_1h

In [20]:
# =========================================================
# Model 11:
# Baseline
# + 최근 24시간 고액거래 횟수
# + 최근 1시간 누적 거래금액
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + 24h High Amount Count + 1h Rolling Amount"
FIXED_THRESHOLD = 0.990239

model11_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "rolling_sum_amt_1h"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model11_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model11_features))
print("사용 변수:", model11_features)


# 2. 설명변수와 목표변수 생성
X_model11 = df[model11_features].copy()
y_model11 = df["is_fraud"].astype("int8").copy()

X_model11["category"] = (
    X_model11["category"]
    .astype("category")
)

print("\n변수 자료형")
print(X_model11.dtypes)

print("\n결측치 수")
print(X_model11.isna().sum())

print("\n추가 변수 기초 통계")
print(
    X_model11[
        [
            "recent_24h_high_amt_count",
            "rolling_sum_amt_1h"
        ]
    ].describe()
)

print("\n무한대 개수")
print(
    np.isinf(
        X_model11[
            [
                "recent_24h_high_amt_count",
                "rolling_sum_amt_1h"
            ]
        ]
    ).sum()
)


# 3. 시간순 70:30 분할
split_index_model11 = int(len(df) * 0.70)

X_train_model11 = (
    X_model11
    .iloc[:split_index_model11]
    .copy()
)

X_valid_model11 = (
    X_model11
    .iloc[split_index_model11:]
    .copy()
)

y_train_model11 = (
    y_model11
    .iloc[:split_index_model11]
    .copy()
)

y_valid_model11 = (
    y_model11
    .iloc[split_index_model11:]
    .copy()
)

print("\nTrain:", X_train_model11.shape)
print("Valid:", X_valid_model11.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model11 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model11 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model11:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model11:,
        "trans_date_trans_time"
    ].max()
)

print(
    "\nTrain 이상거래 건수:",
    int(y_train_model11.sum())
)

print(
    "Valid 이상거래 건수:",
    int(y_valid_model11.sum())
)

print(
    "Train 이상거래율:",
    y_train_model11.mean()
)

print(
    "Valid 이상거래율:",
    y_valid_model11.mean()
)


# 4. 클래스 불균형 가중치 계산
negative_count_model11 = int(
    (y_train_model11 == 0).sum()
)

positive_count_model11 = int(
    (y_train_model11 == 1).sum()
)

scale_pos_weight_model11 = (
    negative_count_model11
    / positive_count_model11
)

print("\n정상거래 수:", negative_count_model11)
print("이상거래 수:", positive_count_model11)
print("scale_pos_weight:", scale_pos_weight_model11)


# 5. 동일 하이퍼파라미터로 모델 생성
model11_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model11
)


# 6. 학습
model11_7030.fit(
    X_train_model11,
    y_train_model11,

    eval_X=X_valid_model11,
    eval_y=y_valid_model11,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model11 = model11_7030.predict_proba(
    X_valid_model11,
    num_iteration=model11_7030.best_iteration_
)[:, 1]


# 8. Baseline 고정 임계값 적용
valid_pred_model11 = (
    valid_prob_model11 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model11 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model11_features),
    "best_iteration": model11_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model11,
        valid_prob_model11
    ),

    "roc_auc": roc_auc_score(
        y_valid_model11,
        valid_prob_model11
    ),

    "precision": precision_score(
        y_valid_model11,
        valid_pred_model11,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model11,
        valid_pred_model11,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model11,
        valid_pred_model11,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 11: "
    "Baseline + 24h High Amount Count "
    "+ 1h Rolling Amount 70:30 =========="
)

print("변수 수         :", result_model11["feature_count"])
print("Best iteration :", result_model11["best_iteration"])
print("고정 임계값     :", round(result_model11["threshold"], 6))
print("PR-AUC         :", round(result_model11["pr_auc"], 6))
print("ROC-AUC        :", round(result_model11["roc_auc"], 6))
print("Precision      :", round(result_model11["precision"], 6))
print("Recall         :", round(result_model11["recall"], 6))
print("F1-score       :", round(result_model11["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model11,
        valid_pred_model11
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model11["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model11["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model11["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model11["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model11["f1"]
        - result_7030["f1"],
        6
    )
)


# 12. Model 8 대비 변화
print(
    "\n========== Model 8 대비 변화 =========="
)

print(
    "PR-AUC 변화:",
    round(
        result_model11["pr_auc"]
        - result_model8["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model11["roc_auc"]
        - result_model8["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model11["precision"]
        - result_model8["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model11["recall"]
        - result_model8["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model11["f1"]
        - result_model8["f1"],
        6
    )
)

사용 변수 수: 6
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'recent_24h_high_amt_count', 'rolling_sum_amt_1h']

변수 자료형
category                     category
amt                           float64
trans_hour                      int64
age                             int64
recent_24h_high_amt_count       int64
rolling_sum_amt_1h            float64
dtype: object

결측치 수
category                     0
amt                          0
trans_hour                   0
age                          0
recent_24h_high_amt_count    0
rolling_sum_amt_1h           0
dtype: int64

추가 변수 기초 통계
       recent_24h_high_amt_count  rolling_sum_amt_1h
count               1.296675e+06        1.296675e+06
mean                4.816936e-02        8.491117e+01
std                 2.741523e-01        1.947081e+02
min                 0.000000e+00        1.000000e+00
25%                 0.000000e+00        1.425000e+01
50%                 0.000000e+00        5.429000e+01
75%                 0.000000e+00        9.630000e+

Model 8보다는 F1이 미세하게 성능 떨어짐. PR-AUC와 Precision은 조금 좋아짐. 

# Model 12. baseline + recent_24h_high_amt_count + amt_to_prior_median_ratio

가장 강력했던 단독 변수를 조합

In [22]:
# =========================================================
# Model 12:
# Baseline
# + 최근 24시간 고액거래 횟수
# + 개인별 평소 금액 대비 거래금액 비율
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + 24h High Amount Count + Personalized Amount Ratio"
FIXED_THRESHOLD = 0.990239

model12_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model12_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model12_features))
print("사용 변수:", model12_features)


# 2. 설명변수와 목표변수 생성
X_model12 = df[model12_features].copy()
y_model12 = df["is_fraud"].astype("int8").copy()

X_model12["category"] = (
    X_model12["category"]
    .astype("category")
)

print("\n변수 자료형")
print(X_model12.dtypes)

print("\n결측치 수")
print(X_model12.isna().sum())

print("\n추가 변수 기초 통계")
print(
    X_model12[
        [
            "recent_24h_high_amt_count",
            "amt_to_prior_median_ratio"
        ]
    ].describe()
)

print("\n무한대 개수")
print(
    np.isinf(
        X_model12[
            [
                "recent_24h_high_amt_count",
                "amt_to_prior_median_ratio"
            ]
        ]
    ).sum()
)


# 3. 시간순 70:30 분할
split_index_model12 = int(len(df) * 0.70)

X_train_model12 = (
    X_model12
    .iloc[:split_index_model12]
    .copy()
)

X_valid_model12 = (
    X_model12
    .iloc[split_index_model12:]
    .copy()
)

y_train_model12 = (
    y_model12
    .iloc[:split_index_model12]
    .copy()
)

y_valid_model12 = (
    y_model12
    .iloc[split_index_model12:]
    .copy()
)

print("\nTrain:", X_train_model12.shape)
print("Valid:", X_valid_model12.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model12 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model12 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model12:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model12:,
        "trans_date_trans_time"
    ].max()
)

print(
    "\nTrain 이상거래 건수:",
    int(y_train_model12.sum())
)

print(
    "Valid 이상거래 건수:",
    int(y_valid_model12.sum())
)

print(
    "Train 이상거래율:",
    y_train_model12.mean()
)

print(
    "Valid 이상거래율:",
    y_valid_model12.mean()
)


# 4. 클래스 불균형 가중치 계산
negative_count_model12 = int(
    (y_train_model12 == 0).sum()
)

positive_count_model12 = int(
    (y_train_model12 == 1).sum()
)

scale_pos_weight_model12 = (
    negative_count_model12
    / positive_count_model12
)

print("\n정상거래 수:", negative_count_model12)
print("이상거래 수:", positive_count_model12)
print("scale_pos_weight:", scale_pos_weight_model12)


# 5. 동일 하이퍼파라미터로 모델 생성
model12_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model12
)


# 6. 학습
model12_7030.fit(
    X_train_model12,
    y_train_model12,

    eval_X=X_valid_model12,
    eval_y=y_valid_model12,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model12 = model12_7030.predict_proba(
    X_valid_model12,
    num_iteration=model12_7030.best_iteration_
)[:, 1]


# 8. Baseline 고정 임계값 적용
valid_pred_model12 = (
    valid_prob_model12 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model12 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model12_features),
    "best_iteration": model12_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model12,
        valid_prob_model12
    ),

    "roc_auc": roc_auc_score(
        y_valid_model12,
        valid_prob_model12
    ),

    "precision": precision_score(
        y_valid_model12,
        valid_pred_model12,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model12,
        valid_pred_model12,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model12,
        valid_pred_model12,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 12: "
    "Baseline + 24h High Amount Count "
    "+ Personalized Amount Ratio 70:30 =========="
)

print("변수 수         :", result_model12["feature_count"])
print("Best iteration :", result_model12["best_iteration"])
print("고정 임계값     :", round(result_model12["threshold"], 6))
print("PR-AUC         :", round(result_model12["pr_auc"], 6))
print("ROC-AUC        :", round(result_model12["roc_auc"], 6))
print("Precision      :", round(result_model12["precision"], 6))
print("Recall         :", round(result_model12["recall"], 6))
print("F1-score       :", round(result_model12["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model12,
        valid_pred_model12
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model12["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model12["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model12["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model12["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model12["f1"]
        - result_7030["f1"],
        6
    )
)


# 12. Model 8 대비 변화
print(
    "\n========== Model 8 대비 변화 =========="
)

print(
    "PR-AUC 변화:",
    round(
        result_model12["pr_auc"]
        - result_model8["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model12["roc_auc"]
        - result_model8["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model12["precision"]
        - result_model8["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model12["recall"]
        - result_model8["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model12["f1"]
        - result_model8["f1"],
        6
    )
)

사용 변수 수: 6
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'recent_24h_high_amt_count', 'amt_to_prior_median_ratio']

변수 자료형
category                     category
amt                           float64
trans_hour                      int64
age                             int64
recent_24h_high_amt_count       int64
amt_to_prior_median_ratio     float64
dtype: object

결측치 수
category                        0
amt                             0
trans_hour                      0
age                             0
recent_24h_high_amt_count       0
amt_to_prior_median_ratio    1649
dtype: int64

추가 변수 기초 통계
       recent_24h_high_amt_count  amt_to_prior_median_ratio
count               1.296675e+06               1.295026e+06
mean                4.816936e-02               1.643860e+00
std                 2.741523e-01               4.558937e+00
min                 0.000000e+00               2.176430e-03
25%                 0.000000e+00               2.606922e-01
50%                 0.000000e+00    

Model 12가 현재 1위.

# Model 13. baseline + amt_to_prior_median_ratio + rolling_sum_amt_1h

In [23]:
# =========================================================
# Model 13:
# Baseline
# + 개인별 평소 금액 대비 거래금액 비율
# + 최근 1시간 누적 거래금액
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + Personalized Amount Ratio + 1h Rolling Amount"
FIXED_THRESHOLD = 0.990239

model13_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model13_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model13_features))
print("사용 변수:", model13_features)


# 2. 설명변수와 목표변수 생성
X_model13 = df[model13_features].copy()
y_model13 = df["is_fraud"].astype("int8").copy()

X_model13["category"] = (
    X_model13["category"]
    .astype("category")
)

print("\n변수 자료형")
print(X_model13.dtypes)

print("\n결측치 수")
print(X_model13.isna().sum())

print("\n추가 변수 기초 통계")
print(
    X_model13[
        [
            "amt_to_prior_median_ratio",
            "rolling_sum_amt_1h"
        ]
    ].describe()
)

print("\n무한대 개수")
print(
    np.isinf(
        X_model13[
            [
                "amt_to_prior_median_ratio",
                "rolling_sum_amt_1h"
            ]
        ]
    ).sum()
)


# 3. 시간순 70:30 분할
split_index_model13 = int(len(df) * 0.70)

X_train_model13 = (
    X_model13
    .iloc[:split_index_model13]
    .copy()
)

X_valid_model13 = (
    X_model13
    .iloc[split_index_model13:]
    .copy()
)

y_train_model13 = (
    y_model13
    .iloc[:split_index_model13]
    .copy()
)

y_valid_model13 = (
    y_model13
    .iloc[split_index_model13:]
    .copy()
)

print("\nTrain:", X_train_model13.shape)
print("Valid:", X_valid_model13.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model13 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model13 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model13:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model13:,
        "trans_date_trans_time"
    ].max()
)

print(
    "\nTrain 이상거래 건수:",
    int(y_train_model13.sum())
)

print(
    "Valid 이상거래 건수:",
    int(y_valid_model13.sum())
)

print(
    "Train 이상거래율:",
    y_train_model13.mean()
)

print(
    "Valid 이상거래율:",
    y_valid_model13.mean()
)


# 4. 클래스 불균형 가중치 계산
negative_count_model13 = int(
    (y_train_model13 == 0).sum()
)

positive_count_model13 = int(
    (y_train_model13 == 1).sum()
)

scale_pos_weight_model13 = (
    negative_count_model13
    / positive_count_model13
)

print("\n정상거래 수:", negative_count_model13)
print("이상거래 수:", positive_count_model13)
print("scale_pos_weight:", scale_pos_weight_model13)


# 5. 동일 하이퍼파라미터로 모델 생성
model13_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model13
)


# 6. 학습
model13_7030.fit(
    X_train_model13,
    y_train_model13,

    eval_X=X_valid_model13,
    eval_y=y_valid_model13,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model13 = model13_7030.predict_proba(
    X_valid_model13,
    num_iteration=model13_7030.best_iteration_
)[:, 1]


# 8. Baseline 고정 임계값 적용
valid_pred_model13 = (
    valid_prob_model13 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model13 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model13_features),
    "best_iteration": model13_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model13,
        valid_prob_model13
    ),

    "roc_auc": roc_auc_score(
        y_valid_model13,
        valid_prob_model13
    ),

    "precision": precision_score(
        y_valid_model13,
        valid_pred_model13,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model13,
        valid_pred_model13,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model13,
        valid_pred_model13,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 13: "
    "Baseline + Personalized Amount Ratio "
    "+ 1h Rolling Amount 70:30 =========="
)

print("변수 수         :", result_model13["feature_count"])
print("Best iteration :", result_model13["best_iteration"])
print("고정 임계값     :", round(result_model13["threshold"], 6))
print("PR-AUC         :", round(result_model13["pr_auc"], 6))
print("ROC-AUC        :", round(result_model13["roc_auc"], 6))
print("Precision      :", round(result_model13["precision"], 6))
print("Recall         :", round(result_model13["recall"], 6))
print("F1-score       :", round(result_model13["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model13,
        valid_pred_model13
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model13["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model13["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model13["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model13["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model13["f1"]
        - result_7030["f1"],
        6
    )
)


# 12. Model 4 대비 변화
print(
    "\n========== Model 4 대비 변화 =========="
)

print(
    "PR-AUC 변화:",
    round(
        result_model13["pr_auc"]
        - result_model4["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model13["roc_auc"]
        - result_model4["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model13["precision"]
        - result_model4["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model13["recall"]
        - result_model4["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model13["f1"]
        - result_model4["f1"],
        6
    )
)


# 13. Model 7 대비 변화
print(
    "\n========== Model 7 대비 변화 =========="
)

print(
    "PR-AUC 변화:",
    round(
        result_model13["pr_auc"]
        - result_model7["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model13["roc_auc"]
        - result_model7["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model13["precision"]
        - result_model7["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model13["recall"]
        - result_model7["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model13["f1"]
        - result_model7["f1"],
        6
    )
)

사용 변수 수: 6
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'amt_to_prior_median_ratio', 'rolling_sum_amt_1h']

변수 자료형
category                     category
amt                           float64
trans_hour                      int64
age                             int64
amt_to_prior_median_ratio     float64
rolling_sum_amt_1h            float64
dtype: object

결측치 수
category                        0
amt                             0
trans_hour                      0
age                             0
amt_to_prior_median_ratio    1649
rolling_sum_amt_1h              0
dtype: int64

추가 변수 기초 통계
       amt_to_prior_median_ratio  rolling_sum_amt_1h
count               1.295026e+06        1.296675e+06
mean                1.643860e+00        8.491117e+01
std                 4.558937e+00        1.947081e+02
min                 2.176430e-03        1.000000e+00
25%                 2.606922e-01        1.425000e+01
50%                 1.003543e+00        5.429000e+01
75%                 1.894498e+00

Model 13 효과 있음. 단독으로 썼을 때보다 모든 핵심 지표가 좋아짐.

# Model 14. 

- recent_24h_high_amt_count,
- amt_to_prior_median_ratio
- rolling_sum_amt_1h

In [ ]:
# =========================================================
# Model 14:
# Baseline
# + 최근 24시간 고액거래 횟수
# + 개인별 평소 금액 대비 거래금액 비율
# + 최근 1시간 누적 거래금액
# 시간순 70:30
# =========================================================

MODEL_NAME = (
    "Baseline + 24h High Amount Count "
    "+ Personalized Amount Ratio "
    "+ 1h Rolling Amount"
)

FIXED_THRESHOLD = 0.990239

model14_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model14_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model14_features))
print("사용 변수:", model14_features)


# 2. 설명변수와 목표변수 생성
X_model14 = df[model14_features].copy()
y_model14 = df["is_fraud"].astype("int8").copy()

X_model14["category"] = (
    X_model14["category"]
    .astype("category")
)

print("\n변수 자료형")
print(X_model14.dtypes)

print("\n결측치 수")
print(X_model14.isna().sum())

print("\n추가 변수 기초 통계")
print(
    X_model14[
        [
            "recent_24h_high_amt_count",
            "amt_to_prior_median_ratio",
            "rolling_sum_amt_1h"
        ]
    ].describe()
)

print("\n무한대 개수")
print(
    np.isinf(
        X_model14[
            [
                "recent_24h_high_amt_count",
                "amt_to_prior_median_ratio",
                "rolling_sum_amt_1h"
            ]
        ]
    ).sum()
)


# 3. 시간순 70:30 분할
split_index_model14 = int(len(df) * 0.70)

X_train_model14 = (
    X_model14
    .iloc[:split_index_model14]
    .copy()
)

X_valid_model14 = (
    X_model14
    .iloc[split_index_model14:]
    .copy()
)

y_train_model14 = (
    y_model14
    .iloc[:split_index_model14]
    .copy()
)

y_valid_model14 = (
    y_model14
    .iloc[split_index_model14:]
    .copy()
)

print("\nTrain:", X_train_model14.shape)
print("Valid:", X_valid_model14.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model14 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model14 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model14:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model14:,
        "trans_date_trans_time"
    ].max()
)

print(
    "\nTrain 이상거래 건수:",
    int(y_train_model14.sum())
)

print(
    "Valid 이상거래 건수:",
    int(y_valid_model14.sum())
)

print(
    "Train 이상거래율:",
    y_train_model14.mean()
)

print(
    "Valid 이상거래율:",
    y_valid_model14.mean()
)


# 4. 클래스 불균형 가중치 계산
negative_count_model14 = int(
    (y_train_model14 == 0).sum()
)

positive_count_model14 = int(
    (y_train_model14 == 1).sum()
)

scale_pos_weight_model14 = (
    negative_count_model14
    / positive_count_model14
)

print("\n정상거래 수:", negative_count_model14)
print("이상거래 수:", positive_count_model14)
print("scale_pos_weight:", scale_pos_weight_model14)


# 5. 동일 하이퍼파라미터로 모델 생성
model14_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model14
)


# 6. 학습
model14_7030.fit(
    X_train_model14,
    y_train_model14,

    eval_X=X_valid_model14,
    eval_y=y_valid_model14,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model14 = model14_7030.predict_proba(
    X_valid_model14,
    num_iteration=model14_7030.best_iteration_
)[:, 1]


# 8. Baseline 고정 임계값 적용
valid_pred_model14 = (
    valid_prob_model14 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model14 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model14_features),
    "best_iteration": model14_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model14,
        valid_prob_model14
    ),

    "roc_auc": roc_auc_score(
        y_valid_model14,
        valid_prob_model14
    ),

    "precision": precision_score(
        y_valid_model14,
        valid_pred_model14,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model14,
        valid_pred_model14,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model14,
        valid_pred_model14,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 14: "
    "Baseline + 24h High Amount Count "
    "+ Personalized Amount Ratio "
    "+ 1h Rolling Amount 70:30 =========="
)

print("변수 수         :", result_model14["feature_count"])
print("Best iteration :", result_model14["best_iteration"])
print("고정 임계값     :", round(result_model14["threshold"], 6))
print("PR-AUC         :", round(result_model14["pr_auc"], 6))
print("ROC-AUC        :", round(result_model14["roc_auc"], 6))
print("Precision      :", round(result_model14["precision"], 6))
print("Recall         :", round(result_model14["recall"], 6))
print("F1-score       :", round(result_model14["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model14,
        valid_pred_model14
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model14["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model14["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model14["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model14["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model14["f1"]
        - result_7030["f1"],
        6
    )
)


# 12. 현재 1위 Model 12 대비 변화
print(
    "\n========== Model 12 대비 변화 =========="
)

print(
    "PR-AUC 변화:",
    round(
        result_model14["pr_auc"]
        - result_model12["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model14["roc_auc"]
        - result_model12["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model14["precision"]
        - result_model12["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model14["recall"]
        - result_model12["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model14["f1"]
        - result_model12["f1"],
        6
    )
)


# 13. Model 11 대비 변화
print(
    "\n========== Model 11 대비 변화 =========="
)

print(
    "PR-AUC 변화:",
    round(
        result_model14["pr_auc"]
        - result_model11["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model14["roc_auc"]
        - result_model11["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model14["precision"]
        - result_model11["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model14["recall"]
        - result_model11["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model14["f1"]
        - result_model11["f1"],
        6
    )
)

사용 변수 수: 7
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'recent_24h_high_amt_count', 'amt_to_prior_median_ratio', 'rolling_sum_amt_1h']

변수 자료형
category                     category
amt                           float64
trans_hour                      int64
age                             int64
recent_24h_high_amt_count       int64
amt_to_prior_median_ratio     float64
rolling_sum_amt_1h            float64
dtype: object

결측치 수
category                        0
amt                             0
trans_hour                      0
age                             0
recent_24h_high_amt_count       0
amt_to_prior_median_ratio    1649
rolling_sum_amt_1h              0
dtype: int64

추가 변수 기초 통계
       recent_24h_high_amt_count  amt_to_prior_median_ratio  \
count               1.296675e+06               1.295026e+06   
mean                4.816936e-02               1.643860e+00   
std                 2.741523e-01               4.558937e+00   
min                 0.000000e+00               2.1

: 

# lightGBM 2차 (0807 금요일 이후)

In [1]:
import pandas as pd
import lightgbm as lgb

csv_path = (
    r"C:\Users\splen\OneDrive\Desktop\BDAI_"
    r"\BOOSTMAP\Fraud-FDS-Project\data"
    r"\fraud_full_features.csv"
)

df = pd.read_csv(csv_path)

print(df.shape)
print(df["category"].dtype)
print(df["amt_zscore_card"].isna().sum())
print(df["amt_zscore_card"].describe())

(1296675, 31)
str
0
count    1.296675e+06
mean     2.923736e-02
std      2.880264e+00
min     -1.308383e+03
25%     -4.327248e-01
50%     -1.895656e-01
75%      1.126558e-01
max      1.323901e+03
Name: amt_zscore_card, dtype: float64


amt_zscore_card는 최솟값 -1308, 최댓값 1324처럼 극단값이 있는데, LightGBM은 트리 모델이라 일단 그대로 넣고 성능을 확인해도 괜찮음.

# Model 15. 

baseline +
- amt_to_prior_median_ratio
- rolling_sum_amt_1h
- recent_24h_high_amt_count <- 여기까지 Model 14.
- amt_zscore_card

데이터를 시간순 정렬

In [4]:
# 거래시간 datetime 변환
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])

# 시간순 정렬 + 인덱스 초기화
df = (
    df.sort_values("trans_date_trans_time")
      .reset_index(drop=True)
)

# 확인
split_idx = int(len(df) * 0.7)

print("전체 시간 정렬 여부:",
      df["trans_date_trans_time"].is_monotonic_increasing)

print("Train fraud count:",
      df.iloc[:split_idx]["is_fraud"].sum())

print("Valid fraud count:",
      df.iloc[split_idx:]["is_fraud"].sum())

print("Train 시작:", df.iloc[0]["trans_date_trans_time"])
print("Train 끝:", df.iloc[split_idx - 1]["trans_date_trans_time"])
print("Valid 시작:", df.iloc[split_idx]["trans_date_trans_time"])
print("Valid 끝:", df.iloc[-1]["trans_date_trans_time"])

전체 시간 정렬 여부: True
Train fraud count: 5121
Valid fraud count: 2385
Train 시작: 2019-01-01 00:00:18
Train 끝: 2019-12-28 17:18:24
Valid 시작: 2019-12-28 17:18:38
Valid 끝: 2020-06-21 12:13:37


In [5]:
import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m15 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "amt_zscore_card"
]

df["category"] = df["category"].astype("category")

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m15]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m15]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

model_m15 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m15.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m15 = model_m15.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m15 = (valid_prob_m15 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m15)
roc_auc = roc_auc_score(y_valid, valid_prob_m15)
precision = precision_score(y_valid, valid_pred_m15)
recall = recall_score(y_valid, valid_pred_m15)
f1 = f1_score(y_valid, valid_pred_m15)
cm = confusion_matrix(y_valid, valid_pred_m15)

print("\n===== Model 15 : M14 + amt_zscore_card =====")
print("Best iteration:", model_m15.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.95429	valid_0's binary_logloss: 0.0285481
[100]	valid_0's average_precision: 0.965145	valid_0's binary_logloss: 0.0181561
[150]	valid_0's average_precision: 0.969583	valid_0's binary_logloss: 0.0134811
[200]	valid_0's average_precision: 0.971398	valid_0's binary_logloss: 0.0111477
[250]	valid_0's average_precision: 0.972429	valid_0's binary_logloss: 0.0095018
[300]	valid_0's average_precision: 0.973531	valid_0's binary_logloss: 0.00839297
[350]	valid_0's average_precision: 0.974281	valid_0's binary_logloss: 0.00754334
[400]	valid_0's average_precision: 0.974605	valid_0's binary_logloss: 0.00688743
[450]	valid_0's average_precision: 0.974779	valid_0's binary_logloss: 0.00630874
[500]	valid_0's average_precision: 0.974994	valid_0's binary_logloss: 0.00585248
[550]	valid_0's average_p

M15는 1위였던 M14의 성능보다 좋음. New 1위

# Model 16. 
M14 + amt_ratio_to_mean 

In [6]:
# ============================================================
# Model 16
# M14 + amt_ratio_to_mean
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m16 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "amt_ratio_to_mean"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m16]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m16]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)
print("amt_ratio_to_mean NaN:", df["amt_ratio_to_mean"].isna().sum())

model_m16 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m16.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m16 = model_m16.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m16 = (valid_prob_m16 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m16)
roc_auc = roc_auc_score(y_valid, valid_prob_m16)
precision = precision_score(y_valid, valid_pred_m16)
recall = recall_score(y_valid, valid_pred_m16)
f1 = f1_score(y_valid, valid_pred_m16)
cm = confusion_matrix(y_valid, valid_pred_m16)

print("\n===== Model 16 : M14 + amt_ratio_to_mean =====")
print("Best iteration:", model_m16.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
amt_ratio_to_mean NaN: 0
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.94719	valid_0's binary_logloss: 0.0294833
[100]	valid_0's average_precision: 0.961264	valid_0's binary_logloss: 0.0191715
[150]	valid_0's average_precision: 0.96768	valid_0's binary_logloss: 0.0145066
[200]	valid_0's average_precision: 0.970528	valid_0's binary_logloss: 0.0119661
[250]	valid_0's average_precision: 0.971813	valid_0's binary_logloss: 0.0101714
[300]	valid_0's average_precision: 0.972962	valid_0's binary_logloss: 0.00892376
[350]	valid_0's average_precision: 0.973397	valid_0's binary_logloss: 0.0080487
[400]	valid_0's average_precision: 0.973762	valid_0's binary_logloss: 0.00730218
[450]	valid_0's average_precision: 0.974011	valid_0's binary_logloss: 0.00669233
[500]	valid_0's average_precision: 0.974345	valid_0's binary_logloss: 0.00624301
[5

M16은 M14보다 좋아졌지만 M15보다는 낮음.
M15랑 M16에서 각각 추가한 것을 조합해서 그 2개를 추가했을때 성능이 어떻게 되는지 보기.

# Model 17.

M14 + amt_zscore_card(M15에 단독추가한것) + amt_ratio_to_mean(M16에 단독추가한것)

In [7]:
# ============================================================
# Model 17
# M14 + amt_zscore_card + amt_ratio_to_mean
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m17 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "amt_zscore_card",
    "amt_ratio_to_mean"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m17]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m17]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

model_m17 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m17.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m17 = model_m17.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m17 = (valid_prob_m17 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m17)
roc_auc = roc_auc_score(y_valid, valid_prob_m17)
precision = precision_score(y_valid, valid_pred_m17)
recall = recall_score(y_valid, valid_pred_m17)
f1 = f1_score(y_valid, valid_pred_m17)
cm = confusion_matrix(y_valid, valid_pred_m17)

print("\n===== Model 17 =====")
print("M14 + amt_zscore_card + amt_ratio_to_mean")
print("Best iteration:", model_m17.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== 현재 1위 M15 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975622:+.6f}")
print(f"Precision : {precision - 0.979205:+.6f}")
print(f"Recall    : {recall - 0.888470:+.6f}")
print(f"F1-score  : {f1 - 0.931633:+.6f}")

Train: (907672, 9)
Valid: (389003, 9)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.938879	valid_0's binary_logloss: 0.028324
[100]	valid_0's average_precision: 0.956621	valid_0's binary_logloss: 0.0179945
[150]	valid_0's average_precision: 0.962572	valid_0's binary_logloss: 0.0136241
[200]	valid_0's average_precision: 0.966331	valid_0's binary_logloss: 0.0110682
[250]	valid_0's average_precision: 0.967928	valid_0's binary_logloss: 0.009392
[300]	valid_0's average_precision: 0.96912	valid_0's binary_logloss: 0.0082461
[350]	valid_0's average_precision: 0.970457	valid_0's binary_logloss: 0.00720781
[400]	valid_0's average_precision: 0.97103	valid_0's binary_logloss: 0.00654551
[450]	valid_0's average_precision: 0.971535	valid_0's binary_logloss: 0.00598835
[500]	valid_0's average_precision: 0.972079	valid_0's binary_logloss: 0.00553295
[550]	valid_0's average_preci

M15, M16보다 성능 떨어짐. 후보에서 제외.

애초에 두 변수의 정의가 비슷함.

- amt_ratio_to_mean = 현재 금액이 고객 과거 평균의 몇 배인가
- amt_zscore_card = 현재 금액이 고객 과거 평균에서 표준편차 기준으로 얼마나 벗어났는가

내가 궁금한 건 결국 두 번째 변수가 이미 첫 번째 변수가 제공하는 정보 외에 추가적인 예측 정보를 주는가?

만약 둘이 서로 아주 좋은 보완 정보를 제공했다면 둘 다 넣었을 때 적어도 M15보다 좋아질 가능성을 기대할 수 있는데, 실제로는 그렇지 않았음.

# Model 18. 

M14 + customer_transaction_count(고객의 이전 거래 횟수)

In [9]:
# ============================================================
# Model 18
# M14 + customer_transaction_count
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m18 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "customer_transaction_count"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m18]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m18]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)
print(
    "customer_transaction_count NaN:",
    df["customer_transaction_count"].isna().sum()
)

model_m18 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m18.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m18 = model_m18.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m18 = (valid_prob_m18 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m18)
roc_auc = roc_auc_score(y_valid, valid_prob_m18)
precision = precision_score(y_valid, valid_pred_m18)
recall = recall_score(y_valid, valid_pred_m18)
f1 = f1_score(y_valid, valid_pred_m18)
cm = confusion_matrix(y_valid, valid_pred_m18)

print("\n===== Model 18 : M14 + customer_transaction_count =====")
print("Best iteration:", model_m18.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== 현재 1위 M15 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975622:+.6f}")
print(f"Precision : {precision - 0.979205:+.6f}")
print(f"Recall    : {recall - 0.888470:+.6f}")
print(f"F1-score  : {f1 - 0.931633:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
customer_transaction_count NaN: 0
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.945596	valid_0's binary_logloss: 0.0232636
[100]	valid_0's average_precision: 0.960453	valid_0's binary_logloss: 0.0129173
[150]	valid_0's average_precision: 0.96607	valid_0's binary_logloss: 0.00873622
[200]	valid_0's average_precision: 0.967158	valid_0's binary_logloss: 0.00712627
[250]	valid_0's average_precision: 0.967928	valid_0's binary_logloss: 0.00594021
[300]	valid_0's average_precision: 0.968753	valid_0's binary_logloss: 0.00529976
[350]	valid_0's average_precision: 0.968928	valid_0's binary_logloss: 0.00468012
Early stopping, best iteration is:
[319]	valid_0's average_precision: 0.969257	valid_0's binary_logloss: 0.00505179
Evaluated only: average_precision

===== Model 18 : M14 + customer_transaction_count =====
Best iteration: 319
PR-A

고객의 '이전 이상거래' 횟수가 아니라, 그냥 '이전 전체 거래' 횟수라서 개선 안될 것이라 예상했고,

역시나 Recall이 크게 무너짐(0.79)

# Model 19.

M14 + customer_mean_amt(해당 고객의 과거 평균 거래금액)

In [10]:
# ============================================================
# Model 19
# M14 + customer_mean_amt
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m19 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "customer_mean_amt"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m19]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m19]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)
print("customer_mean_amt NaN:", df["customer_mean_amt"].isna().sum())

model_m19 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m19.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m19 = model_m19.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m19 = (valid_prob_m19 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m19)
roc_auc = roc_auc_score(y_valid, valid_prob_m19)
precision = precision_score(y_valid, valid_pred_m19)
recall = recall_score(y_valid, valid_pred_m19)
f1 = f1_score(y_valid, valid_pred_m19)
cm = confusion_matrix(y_valid, valid_pred_m19)

print("\n===== Model 19 : M14 + customer_mean_amt =====")
print("Best iteration:", model_m19.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== 현재 1위 M15 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975622:+.6f}")
print(f"Precision : {precision - 0.979205:+.6f}")
print(f"Recall    : {recall - 0.888470:+.6f}")
print(f"F1-score  : {f1 - 0.931633:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
customer_mean_amt NaN: 0
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.93995	valid_0's binary_logloss: 0.0285585
[100]	valid_0's average_precision: 0.960219	valid_0's binary_logloss: 0.0184484
[150]	valid_0's average_precision: 0.967081	valid_0's binary_logloss: 0.0136635
[200]	valid_0's average_precision: 0.969954	valid_0's binary_logloss: 0.0110988
[250]	valid_0's average_precision: 0.971306	valid_0's binary_logloss: 0.00926865
[300]	valid_0's average_precision: 0.972383	valid_0's binary_logloss: 0.0081763
[350]	valid_0's average_precision: 0.972957	valid_0's binary_logloss: 0.00730143
[400]	valid_0's average_precision: 0.973051	valid_0's binary_logloss: 0.00660038
Early stopping, best iteration is:
[363]	valid_0's average_precision: 0.9732	valid_0's binary_logloss: 0.00709109
Evaluated only: average_precision

===== Model 1

M19도 customer_mean_amt 단독 추가변수로는 우선순위가 낮음

중요한 점은,

M18, M19 모두 고객의 '기본 거래 특성'을 넣었을 때 precision은 올라가고 recall은 떨어지는 패턴나옴. 일반화할 순 없지만, 이 계열의 변수가 모델을 조금 더 보수적으로 만드는 경향이 있을 가능성은 있음.

# Model 20.

M14 + customer_std_amt(고객별 평소 거래금액 변동성)

In [11]:
# ============================================================
# Model 20
# M14 + customer_std_amt
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m20 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "customer_std_amt"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m20]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m20]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)
print("customer_std_amt NaN:", df["customer_std_amt"].isna().sum())

model_m20 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m20.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m20 = model_m20.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m20 = (valid_prob_m20 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m20)
roc_auc = roc_auc_score(y_valid, valid_prob_m20)
precision = precision_score(y_valid, valid_pred_m20)
recall = recall_score(y_valid, valid_pred_m20)
f1 = f1_score(y_valid, valid_pred_m20)
cm = confusion_matrix(y_valid, valid_pred_m20)

print("\n===== Model 20 : M14 + customer_std_amt =====")
print("Best iteration:", model_m20.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== 현재 1위 M15 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975622:+.6f}")
print(f"Precision : {precision - 0.979205:+.6f}")
print(f"Recall    : {recall - 0.888470:+.6f}")
print(f"F1-score  : {f1 - 0.931633:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
customer_std_amt NaN: 0
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.954614	valid_0's binary_logloss: 0.0290485
[100]	valid_0's average_precision: 0.965327	valid_0's binary_logloss: 0.0184522
[150]	valid_0's average_precision: 0.970221	valid_0's binary_logloss: 0.0136161
[200]	valid_0's average_precision: 0.972197	valid_0's binary_logloss: 0.0111689
[250]	valid_0's average_precision: 0.973381	valid_0's binary_logloss: 0.00928316
[300]	valid_0's average_precision: 0.974351	valid_0's binary_logloss: 0.00807987
[350]	valid_0's average_precision: 0.975023	valid_0's binary_logloss: 0.0071265
[400]	valid_0's average_precision: 0.974901	valid_0's binary_logloss: 0.00653582
Early stopping, best iteration is:
[378]	valid_0's average_precision: 0.975228	valid_0's binary_logloss: 0.00672049
Evaluated only: average_precision

===== Model

customer_std_amt(고객별 과거 거래금액 변동성)은 추가 정보를 어느 정도 줬다고 볼 수 있음. M14보다 PR-AUC가 살짝 높아짐. 단, M15보단 낮음.

# Model 21.

M14 + interact_repeat_category(최근 24시간 고액거래 횟수 × 업종별 최근 7일 사기율 (상호작용 변수))



In [12]:
# ============================================================
# Model 21
# M14 + interact_repeat_category
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m21 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "interact_repeat_category"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m21]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m21]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)
print(
    "interact_repeat_category NaN:",
    df["interact_repeat_category"].isna().sum()
)

model_m21 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m21.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m21 = model_m21.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m21 = (valid_prob_m21 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m21)
roc_auc = roc_auc_score(y_valid, valid_prob_m21)
precision = precision_score(y_valid, valid_pred_m21)
recall = recall_score(y_valid, valid_pred_m21)
f1 = f1_score(y_valid, valid_pred_m21)
cm = confusion_matrix(y_valid, valid_pred_m21)

print("\n===== Model 21 : M14 + interact_repeat_category =====")
print("Best iteration:", model_m21.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== 현재 1위 M15 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975622:+.6f}")
print(f"Precision : {precision - 0.979205:+.6f}")
print(f"Recall    : {recall - 0.888470:+.6f}")
print(f"F1-score  : {f1 - 0.931633:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
interact_repeat_category NaN: 0
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.94627	valid_0's binary_logloss: 0.0304117
[100]	valid_0's average_precision: 0.963702	valid_0's binary_logloss: 0.0199432
[150]	valid_0's average_precision: 0.968109	valid_0's binary_logloss: 0.0153838
[200]	valid_0's average_precision: 0.970747	valid_0's binary_logloss: 0.0126891
[250]	valid_0's average_precision: 0.971651	valid_0's binary_logloss: 0.0108855
[300]	valid_0's average_precision: 0.972467	valid_0's binary_logloss: 0.00964257
[350]	valid_0's average_precision: 0.97308	valid_0's binary_logloss: 0.00865047
[400]	valid_0's average_precision: 0.973093	valid_0's binary_logloss: 0.00793926
Early stopping, best iteration is:
[368]	valid_0's average_precision: 0.973159	valid_0's binary_logloss: 0.00838881
Evaluated only: average_precision

=====

M21은 거의 변화가 없음. 

그리고 애초에 상호작용 변수를 보는 건데, 애초에 lightGBM은 비선형 상호작용을 자체적으로 학습하기 때문에, 진짜 필요한지 보려면 다음 두 모델을 비교해야 함.
- M21: M14 + interact_repeat_category
- 새 실험: M14 + category_recent_fraud_rate


# M22 = M14 + category_recent_fraud_ratev (위 추측을 증명하기 위한 실험)

In [13]:
# ============================================================
# Model 22
# M14 + category_recent_fraud_rate
#
# 즉:
# recent_24h_high_amt_count + category_recent_fraud_rate
# 두 원변수를 직접 투입
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m22 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",      # 원변수 1
    "category_recent_fraud_rate"      # 원변수 2
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m22]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m22]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)
print(
    "category_recent_fraud_rate NaN:",
    df["category_recent_fraud_rate"].isna().sum()
)

model_m22 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m22.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m22 = model_m22.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m22 = (valid_prob_m22 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m22)
roc_auc = roc_auc_score(y_valid, valid_prob_m22)
precision = precision_score(y_valid, valid_pred_m22)
recall = recall_score(y_valid, valid_pred_m22)
f1 = f1_score(y_valid, valid_pred_m22)
cm = confusion_matrix(y_valid, valid_pred_m22)

print("\n===== Model 22 =====")
print("M14 + category_recent_fraud_rate")
print("= 두 원변수 직접 투입")
print("Best iteration:", model_m22.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M21(수동 interaction) 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973159:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999420:+.6f}")
print(f"Precision : {precision - 0.967033:+.6f}")
print(f"Recall    : {recall - 0.885535:+.6f}")
print(f"F1-score  : {f1 - 0.924491:+.6f}")

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
category_recent_fraud_rate NaN: 0
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.95406	valid_0's binary_logloss: 0.030569
[100]	valid_0's average_precision: 0.963627	valid_0's binary_logloss: 0.0202929
[150]	valid_0's average_precision: 0.967187	valid_0's binary_logloss: 0.015606
[200]	valid_0's average_precision: 0.969313	valid_0's binary_logloss: 0.0128682
[250]	valid_0's average_precision: 0.970361	valid_0's binary_logloss: 0.0109275
[300]	valid_0's average_precision: 0.971387	valid_0's binary_logloss: 0.00957965
[350]	valid_0's average_precision: 0.972037	valid_0's binary_logloss: 0.00853996
[400]	valid_0's average_precision: 0.972423	valid_0's binary_logloss: 0.0077237
[450]	valid_0's average_precision: 0.972729	valid_0's binary_logloss: 0.00709243
[500]	valid_0's average_precision: 0.973107	valid_0's binary_logloss: 0.006

상호작용 변수 vs 두 원변수 비교결과

상호작용 변수 넣는 게 더 별로. 뚜렷한 성능 향상이 나타나지 않음. 원변수 직접 투입한 모델이 소폭 우수.

하지만, M22도 M14보다는 PR-AUC가 낮아짐. 즉, category_recent_fraud_rate를 추가하는 것 자체가 현재 M14에 큰 도움이 된다고 보기는 어려움.

M22 일단 보류.

# M23 : M14 + merchant_change_count(이전 거래 대비 가맹점이 바뀌었는지 여부)

예상: 애초에 merchant_change_count 는 이상거래 탐지에 영향을 줄 수 없을 것 같음. 당연히 여러 장소에서 소비를 할 거니까 가맹점 변경 여부는 딱히?

In [14]:
# ============================================================
# Model 23
# M14 + merchant_change_count
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m23 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "merchant_change_count"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m23]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m23]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)
print(
    "merchant_change_count NaN:",
    df["merchant_change_count"].isna().sum()
)

model_m23 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m23.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m23 = model_m23.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m23 = (valid_prob_m23 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m23)
roc_auc = roc_auc_score(y_valid, valid_prob_m23)
precision = precision_score(y_valid, valid_pred_m23)
recall = recall_score(y_valid, valid_pred_m23)
f1 = f1_score(y_valid, valid_pred_m23)
cm = confusion_matrix(y_valid, valid_pred_m23)

print("\n===== Model 23 : M14 + merchant_change_count =====")
print("Best iteration:", model_m23.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== 현재 1위 M15 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975622:+.6f}")
print(f"Precision : {precision - 0.979205:+.6f}")
print(f"Recall    : {recall - 0.888470:+.6f}")
print(f"F1-score  : {f1 - 0.931633:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
merchant_change_count NaN: 0
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.941443	valid_0's binary_logloss: 0.0304659
[100]	valid_0's average_precision: 0.959081	valid_0's binary_logloss: 0.0200888
[150]	valid_0's average_precision: 0.966935	valid_0's binary_logloss: 0.0153418
[200]	valid_0's average_precision: 0.969932	valid_0's binary_logloss: 0.0128291
[250]	valid_0's average_precision: 0.971329	valid_0's binary_logloss: 0.0109889
[300]	valid_0's average_precision: 0.972257	valid_0's binary_logloss: 0.00977393
[350]	valid_0's average_precision: 0.972914	valid_0's binary_logloss: 0.00884542
[400]	valid_0's average_precision: 0.973235	valid_0's binary_logloss: 0.00809649
[450]	valid_0's average_precision: 0.973515	valid_0's binary_logloss: 0.00745266
[500]	valid_0's average_precision: 0.973947	valid_0's binary_logloss: 0.0068

예상과는 다르게, M14보다 성능지표가 전부 개선됨. 하지만 M15보다는 낮음. 나중에 다른 변수들이랑 더 조합해서 비교해보기.

# M24 = M14 + has_prior_normal_transaction(현재 거래 이전에 정상거래 이력 여부(0/1)
)

In [15]:
# ============================================================
# Model 24
# M14 + has_prior_normal_transaction
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m24 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "has_prior_normal_transaction"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m24]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m24]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)
print(
    "has_prior_normal_transaction NaN:",
    df["has_prior_normal_transaction"].isna().sum()
)

model_m24 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m24.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m24 = model_m24.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m24 = (valid_prob_m24 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m24)
roc_auc = roc_auc_score(y_valid, valid_prob_m24)
precision = precision_score(y_valid, valid_pred_m24)
recall = recall_score(y_valid, valid_pred_m24)
f1 = f1_score(y_valid, valid_pred_m24)
cm = confusion_matrix(y_valid, valid_pred_m24)

print("\n===== Model 24 : M14 + has_prior_normal_transaction =====")
print("Best iteration:", model_m24.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== 현재 1위 M15 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975622:+.6f}")
print(f"Precision : {precision - 0.979205:+.6f}")
print(f"Recall    : {recall - 0.888470:+.6f}")
print(f"F1-score  : {f1 - 0.931633:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
has_prior_normal_transaction NaN: 0
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.950513	valid_0's binary_logloss: 0.0288175
[100]	valid_0's average_precision: 0.963941	valid_0's binary_logloss: 0.0191609
[150]	valid_0's average_precision: 0.968161	valid_0's binary_logloss: 0.0150404
[200]	valid_0's average_precision: 0.970476	valid_0's binary_logloss: 0.0125381
[250]	valid_0's average_precision: 0.971651	valid_0's binary_logloss: 0.0107772
[300]	valid_0's average_precision: 0.972516	valid_0's binary_logloss: 0.00963537
[350]	valid_0's average_precision: 0.973332	valid_0's binary_logloss: 0.00865919
[400]	valid_0's average_precision: 0.973853	valid_0's binary_logloss: 0.00790594
[450]	valid_0's average_precision: 0.973936	valid_0's binary_logloss: 0.00730851
[500]	valid_0's average_precision: 0.974417	valid_0's binary_logloss:

M24는 꽤 괜찮은 변수. M15를 완전히 넘지는 못했지만, 여태 모델들 중 Recall이 제일 높은 모델임.

# Model 25 = M14 + amt_zscore_card + has_prior_normal_transaction

In [16]:
# ============================================================
# Model 25
# M14 + amt_zscore_card + has_prior_normal_transaction
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m25 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "amt_zscore_card",
    "has_prior_normal_transaction"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m25]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m25]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

model_m25 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m25.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m25 = model_m25.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m25 = (valid_prob_m25 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m25)
roc_auc = roc_auc_score(y_valid, valid_prob_m25)
precision = precision_score(y_valid, valid_pred_m25)
recall = recall_score(y_valid, valid_pred_m25)
f1 = f1_score(y_valid, valid_pred_m25)
cm = confusion_matrix(y_valid, valid_pred_m25)

print("\n===== Model 25 =====")
print("M14 + amt_zscore_card + has_prior_normal_transaction")
print("Best iteration:", model_m25.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== 현재 1위 M15 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975622:+.6f}")
print(f"Precision : {precision - 0.979205:+.6f}")
print(f"Recall    : {recall - 0.888470:+.6f}")
print(f"F1-score  : {f1 - 0.931633:+.6f}")

print("\n===== M24 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.974690:+.6f}")
print(f"Precision : {precision - 0.973829:+.6f}")
print(f"Recall    : {recall - 0.889308:+.6f}")
print(f"F1-score  : {f1 - 0.929652:+.6f}")

Train: (907672, 9)
Valid: (389003, 9)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.933832	valid_0's binary_logloss: 0.0297089
[100]	valid_0's average_precision: 0.960148	valid_0's binary_logloss: 0.0189357
[150]	valid_0's average_precision: 0.963976	valid_0's binary_logloss: 0.014276
[200]	valid_0's average_precision: 0.966728	valid_0's binary_logloss: 0.0119534
[250]	valid_0's average_precision: 0.968724	valid_0's binary_logloss: 0.0100563
[300]	valid_0's average_precision: 0.970036	valid_0's binary_logloss: 0.00876158
[350]	valid_0's average_precision: 0.971152	valid_0's binary_logloss: 0.00773647
[400]	valid_0's average_precision: 0.971803	valid_0's binary_logloss: 0.00700053
[450]	valid_0's average_precision: 0.972261	valid_0's binary_logloss: 0.0064127
[500]	valid_0's average_precision: 0.972705	valid_0's binary_logloss: 0.00591699
[550]	valid_0's average_pr

① M14에 하나씩 추가 → 단독 효과 확인 → ② 성능 개선된 변수 중 의미가 다른 것끼리 조합 → ③ 조합이 더 좋아지는지 확인.

M25는 조합 효과가 없었음. amt_zscore_card와 has_prior_normal_transaction은 각각 단독으로는 효과가 있었지만, 같이 넣었을 때 서로 보완되지는 않았음.

# Model 26 = M15 + merchant_change_count

amt_zscore_card는 금액 이상성, merchant_change_count는 가맹점 변화 패턴이라 정보 성격이 다르고, 둘 다 M14에 단독 추가했을 때 성능이 개선됨. 그래서 둘을 같이 넣었을 때 서로 보완되는지 확인하는 실험.

In [17]:
# ============================================================
# Model 26
# M15 + merchant_change_count
# = M14 + amt_zscore_card + merchant_change_count
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m26 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "amt_zscore_card",
    "merchant_change_count"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m26]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m26]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

model_m26 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m26.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m26 = model_m26.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m26 = (valid_prob_m26 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m26)
roc_auc = roc_auc_score(y_valid, valid_prob_m26)
precision = precision_score(y_valid, valid_pred_m26)
recall = recall_score(y_valid, valid_pred_m26)
f1 = f1_score(y_valid, valid_pred_m26)
cm = confusion_matrix(y_valid, valid_pred_m26)

print("\n===== Model 26 =====")
print("M15 + merchant_change_count")
print("Best iteration:", model_m26.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M15 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975622:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999385:+.6f}")
print(f"Precision : {precision - 0.979205:+.6f}")
print(f"Recall    : {recall - 0.888470:+.6f}")
print(f"F1-score  : {f1 - 0.931633:+.6f}")

print("\n===== M23 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.974515:+.6f}")
print(f"Precision : {precision - 0.974217:+.6f}")
print(f"Recall    : {recall - 0.887212:+.6f}")
print(f"F1-score  : {f1 - 0.928681:+.6f}")

Train: (907672, 9)
Valid: (389003, 9)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.929696	valid_0's binary_logloss: 0.028816
[100]	valid_0's average_precision: 0.954687	valid_0's binary_logloss: 0.0184121
[150]	valid_0's average_precision: 0.962819	valid_0's binary_logloss: 0.0138024
[200]	valid_0's average_precision: 0.967014	valid_0's binary_logloss: 0.0113087
[250]	valid_0's average_precision: 0.96947	valid_0's binary_logloss: 0.00965596
[300]	valid_0's average_precision: 0.970748	valid_0's binary_logloss: 0.00848101
[350]	valid_0's average_precision: 0.972006	valid_0's binary_logloss: 0.00750335
[400]	valid_0's average_precision: 0.972669	valid_0's binary_logloss: 0.00682737
[450]	valid_0's average_precision: 0.973066	valid_0's binary_logloss: 0.00621197
[500]	valid_0's average_precision: 0.973561	valid_0's binary_logloss: 0.00575343
[550]	valid_0's average_p

M26은 M15를 넘지 못함. merchant_change_count 단독으로는 괜찮았지만 amt_zscore_card와 같이 넣었을 때 추가 이득이 없었음.

# Model 27 = M14 + prior_normal_median_amt

amt_to_prior_median_ratio가 이미 M14에 들어가 있지만, 그건 현재 금액이 과거 정상 중앙값의 몇 배인지만 알려줌. 

반면 prior_normal_median_amt는 그 고객의 정상적인 거래금액 수준 자체를 직접 알려주는 변수.

질문: "평소 대비 몇 배인가?”뿐 아니라 “평소 금액 수준 자체”까지 주면 성능이 좋아지는가?

비율만으로는 같은 2배라도 기준금액이 10만 원인지 100만 원인지 구분이 안 되니까, 둘을 같이 주면 추가 정보가 생기는지 확인하는 실험.


In [18]:
# ============================================================
# Model 27
# M14 + prior_normal_median_amt
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m27 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "prior_normal_median_amt"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m27]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m27]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)
print(
    "prior_normal_median_amt NaN:",
    df["prior_normal_median_amt"].isna().sum()
)

model_m27 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m27.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m27 = model_m27.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m27 = (valid_prob_m27 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m27)
roc_auc = roc_auc_score(y_valid, valid_prob_m27)
precision = precision_score(y_valid, valid_pred_m27)
recall = recall_score(y_valid, valid_pred_m27)
f1 = f1_score(y_valid, valid_pred_m27)
cm = confusion_matrix(y_valid, valid_pred_m27)

print("\n===== Model 27 : M14 + prior_normal_median_amt =====")
print("Best iteration:", model_m27.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== 현재 1위 M15 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975622:+.6f}")
print(f"Precision : {precision - 0.979205:+.6f}")
print(f"Recall    : {recall - 0.888470:+.6f}")
print(f"F1-score  : {f1 - 0.931633:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
prior_normal_median_amt NaN: 1649
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.94567	valid_0's binary_logloss: 0.0273438
[100]	valid_0's average_precision: 0.963753	valid_0's binary_logloss: 0.0168236
[150]	valid_0's average_precision: 0.969393	valid_0's binary_logloss: 0.0123872
[200]	valid_0's average_precision: 0.971692	valid_0's binary_logloss: 0.00994391
[250]	valid_0's average_precision: 0.972806	valid_0's binary_logloss: 0.00835773
[300]	valid_0's average_precision: 0.974109	valid_0's binary_logloss: 0.00721774
[350]	valid_0's average_precision: 0.974702	valid_0's binary_logloss: 0.00635467
[400]	valid_0's average_precision: 0.97484	valid_0's binary_logloss: 0.00569165
[450]	valid_0's average_precision: 0.975041	valid_0's binary_logloss: 0.00516415
[500]	valid_0's average_precision: 0.975297	valid_0's binary_logloss: 0

M27은 꽤 의미있음. PR-AUC가 M15를 아주 조금 넘음. 모든 성능지표가 그렇진않지만. 일단.

# Model 28 = M15 + prior_normal_median_amt

[실험 이유]
- amt_zscore_card는 고객 평균/표준편차 기준으로 현재 금액이 얼마나 이상?한지,
- prior_normal_median_amt는 과거 정상거래의 대표 금액 수준 자체 

둘 다 단독 추가에서 강했고 계산 기준도 달라서, 같이 넣었을 때 서로 보완되는지 확인할 가치가 있음.

In [19]:
# ============================================================
# Model 28
# M15 + prior_normal_median_amt
# = M14 + amt_zscore_card + prior_normal_median_amt
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m28 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "amt_zscore_card",
    "prior_normal_median_amt"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m28]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m28]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)
print(
    "prior_normal_median_amt NaN:",
    df["prior_normal_median_amt"].isna().sum()
)

model_m28 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m28.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m28 = model_m28.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m28 = (valid_prob_m28 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m28)
roc_auc = roc_auc_score(y_valid, valid_prob_m28)
precision = precision_score(y_valid, valid_pred_m28)
recall = recall_score(y_valid, valid_pred_m28)
f1 = f1_score(y_valid, valid_pred_m28)
cm = confusion_matrix(y_valid, valid_pred_m28)

print("\n===== Model 28 =====")
print("M15 + prior_normal_median_amt")
print("Best iteration:", model_m28.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M15 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975622:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999385:+.6f}")
print(f"Precision : {precision - 0.979205:+.6f}")
print(f"Recall    : {recall - 0.888470:+.6f}")
print(f"F1-score  : {f1 - 0.931633:+.6f}")

print("\n===== M27 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975753:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999323:+.6f}")
print(f"Precision : {precision - 0.986391:+.6f}")
print(f"Recall    : {recall - 0.881342:+.6f}")
print(f"F1-score  : {f1 - 0.930912:+.6f}")

Train: (907672, 9)
Valid: (389003, 9)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
prior_normal_median_amt NaN: 1649
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.946273	valid_0's binary_logloss: 0.0274368
[100]	valid_0's average_precision: 0.963215	valid_0's binary_logloss: 0.0166808
[150]	valid_0's average_precision: 0.967825	valid_0's binary_logloss: 0.0121871
[200]	valid_0's average_precision: 0.969626	valid_0's binary_logloss: 0.0097016
[250]	valid_0's average_precision: 0.971006	valid_0's binary_logloss: 0.00796405
[300]	valid_0's average_precision: 0.971851	valid_0's binary_logloss: 0.00686977
[350]	valid_0's average_precision: 0.97261	valid_0's binary_logloss: 0.00601468
[400]	valid_0's average_precision: 0.973223	valid_0's binary_logloss: 0.00546941
[450]	valid_0's average_precision: 0.973758	valid_0's binary_logloss: 0.0049903
[500]	valid_0's average_precision: 0.974149	valid_0's binary_logloss: 0.

M28은 M15랑 M27 둘 다 확실히 넘지 못함.

-> amt_zscore_card와 prior_normal_median_amt를 같이 넣었을 때 단독 최고 성능들을 합친 시너지는 없었음.

# Model 29 = M14 + is_10x_prior_median

질문 : 연속형 비율 정보가 이미 있어도, “10배 이상”이라는 극단적 이상거래 경계를 별도 이진 변수로 주면 추가 도움이 되는가?

예상 : 딱히

In [20]:
# ============================================================
# Model 29
# M14 + is_10x_prior_median
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m29 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "is_10x_prior_median"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m29]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m29]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)
print(
    "is_10x_prior_median NaN:",
    df["is_10x_prior_median"].isna().sum()
)
print(
    "is_10x_prior_median value counts:\n",
    df["is_10x_prior_median"].value_counts(dropna=False)
)

model_m29 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m29.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m29 = model_m29.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m29 = (valid_prob_m29 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m29)
roc_auc = roc_auc_score(y_valid, valid_prob_m29)
precision = precision_score(y_valid, valid_pred_m29)
recall = recall_score(y_valid, valid_pred_m29)
f1 = f1_score(y_valid, valid_pred_m29)
cm = confusion_matrix(y_valid, valid_pred_m29)

print("\n===== Model 29 : M14 + is_10x_prior_median =====")
print("Best iteration:", model_m29.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== M15 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975622:+.6f}")
print(f"Precision : {precision - 0.979205:+.6f}")
print(f"Recall    : {recall - 0.888470:+.6f}")
print(f"F1-score  : {f1 - 0.931633:+.6f}")

print("\n===== M27 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975753:+.6f}")
print(f"Precision : {precision - 0.986391:+.6f}")
print(f"Recall    : {recall - 0.881342:+.6f}")
print(f"F1-score  : {f1 - 0.930912:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
is_10x_prior_median NaN: 0
is_10x_prior_median value counts:
 is_10x_prior_median
0    1279576
1      17099
Name: count, dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.951618	valid_0's binary_logloss: 0.0291733
[100]	valid_0's average_precision: 0.964204	valid_0's binary_logloss: 0.0193468
[150]	valid_0's average_precision: 0.968661	valid_0's binary_logloss: 0.0148951
[200]	valid_0's average_precision: 0.971206	valid_0's binary_logloss: 0.0123855
[250]	valid_0's average_precision: 0.972086	valid_0's binary_logloss: 0.0106642
[300]	valid_0's average_precision: 0.973001	valid_0's binary_logloss: 0.0094459
[350]	valid_0's average_precision: 0.973638	valid_0's binary_logloss: 0.00847421
[400]	valid_0's average_precision: 0.97398	valid_0's binary_logloss: 0.00779112
[450]	valid_0's average_precision: 0.974151	valid_0's 

M29는 M14보다는 개선됐지만, M15나 M27보다는 약함.

그리고 value_counts를 보면 1이 17,099건이라 전체의 약 1.3% 정도라 너무 희귀해서 못 쓰는 변수 수준은 아님. 다만 이미 M14에 amt_to_prior_median_ratio가 있어서, 10배 이상 여부가 추가로 주는 정보가 제한적이었을 가능성은 있음. 이건 “중복 확정”은 아니고 추가효과가 크지 않았다 정도가 정확함.

# Model 30 = M14 + repeat3(최근 30분 내 거래가 3회 이상인지 여부(0/1))

실험이유 : 시간범위와 의미가 달라서, 짧은 시간 내 급격한 거래 반복이 추가 위험 신호가 되는지 볼 수 있음.

질문 : 24시간 고액거래 빈도 외에, 30분 내 반복거래 폭증까지 같이 보면 탐지력이 좋아지는가?


In [21]:
# ============================================================
# Model 30
# M14 + Repeat3
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m30 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "Repeat3"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m30]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m30]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)
print("Repeat3 NaN:", df["Repeat3"].isna().sum())
print(
    "Repeat3 value counts:\n",
    df["Repeat3"].value_counts(dropna=False)
)

model_m30 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m30.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m30 = model_m30.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m30 = (valid_prob_m30 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m30)
roc_auc = roc_auc_score(y_valid, valid_prob_m30)
precision = precision_score(y_valid, valid_pred_m30)
recall = recall_score(y_valid, valid_pred_m30)
f1 = f1_score(y_valid, valid_pred_m30)
cm = confusion_matrix(y_valid, valid_pred_m30)

print("\n===== Model 30 : M14 + Repeat3 =====")
print("Best iteration:", model_m30.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== M15 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975622:+.6f}")
print(f"Precision : {precision - 0.979205:+.6f}")
print(f"Recall    : {recall - 0.888470:+.6f}")
print(f"F1-score  : {f1 - 0.931633:+.6f}")

print("\n===== M27 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975753:+.6f}")
print(f"Precision : {precision - 0.986391:+.6f}")
print(f"Recall    : {recall - 0.881342:+.6f}")
print(f"F1-score  : {f1 - 0.930912:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
Repeat3 NaN: 0
Repeat3 value counts:
 Repeat3
0    1291393
3       4939
4        321
5         22
Name: count, dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.951778	valid_0's binary_logloss: 0.0292195
[100]	valid_0's average_precision: 0.96376	valid_0's binary_logloss: 0.0195332
[150]	valid_0's average_precision: 0.968354	valid_0's binary_logloss: 0.0150499
[200]	valid_0's average_precision: 0.97087	valid_0's binary_logloss: 0.0125357
[250]	valid_0's average_precision: 0.97216	valid_0's binary_logloss: 0.0107509
[300]	valid_0's average_precision: 0.973154	valid_0's binary_logloss: 0.00956917
[350]	valid_0's average_precision: 0.973781	valid_0's binary_logloss: 0.00859934
[400]	valid_0's average_precision: 0.974198	valid_0's binary_logloss: 0.0078288
[450]	valid_0's average_precision: 0.974301	valid_0's binary_loglo

M30도 M14보다는 좋아졌지만, M15/M27을 넘지는 못함.

Repeat3 분포를 보면 실제로는 꽤 희소함.
- 0: 1,291,393건
- 3 이상: 5,282건 정도

그래도 M14 대비 PR-AUC가 +0.001510, Recall이 +0.002935 오른 걸 보면 특정 소수 거래에서 유용한 신호를 주는 변수일 가능성은 있음. 다만 전체 성능을 크게 끌어올릴 정도는 아니었음.

# Model 31 = M14 + count_30min

실험 이유 : 
Repeat3는 30분 내 반복거래를 임계값으로 요약한 변수라 정보가 압축돼 있음. 반면 count_30min은 최근 30분 내 실제 거래 횟수 자체를 쓰니까, LightGBM이 “3회 이상” 같은 고정 기준 없이 거래 빈도 패턴을 더 유연하게 학습할 수 있는지 보는 실험.

In [22]:
# ============================================================
# Model 31
# M14 + count_30min
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m31 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "count_30min"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m31]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m31]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("count_30min NaN:", df["count_30min"].isna().sum())
print("\ncount_30min 분포:")
print(df["count_30min"].describe())

model_m31 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m31.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m31 = model_m31.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m31 = (valid_prob_m31 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m31)
roc_auc = roc_auc_score(y_valid, valid_prob_m31)
precision = precision_score(y_valid, valid_pred_m31)
recall = recall_score(y_valid, valid_pred_m31)
f1 = f1_score(y_valid, valid_pred_m31)
cm = confusion_matrix(y_valid, valid_pred_m31)

print("\n===== Model 31 : M14 + count_30min =====")
print("Best iteration:", model_m31.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== M30(Repeat3) 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975166:+.6f}")
print(f"Precision : {precision - 0.973733:+.6f}")
print(f"Recall    : {recall - 0.885954:+.6f}")
print(f"F1-score  : {f1 - 0.927772:+.6f}")

print("\n===== 현재 주요 후보 대비 =====")
print(f"M15 대비 PR-AUC : {pr_auc - 0.975622:+.6f}")
print(f"M27 대비 PR-AUC : {pr_auc - 0.975753:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
count_30min NaN: 0

count_30min 분포:
count    1.296675e+06
mean     9.095595e-01
std      4.783908e-01
min      0.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      5.000000e+00
Name: count_30min, dtype: float64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.954918	valid_0's binary_logloss: 0.0303307
[100]	valid_0's average_precision: 0.964204	valid_0's binary_logloss: 0.0202372
[150]	valid_0's average_precision: 0.969181	valid_0's binary_logloss: 0.0153096
[200]	valid_0's average_precision: 0.971191	valid_0's binary_logloss: 0.0126669
[250]	valid_0's average_precision: 0.971962	valid_0's binary_logloss: 0.0108977
[300]	valid_0's average_precision: 0.972983	valid_0's binary_logloss: 0.00967631
[350]	valid_0's average_precision: 0.973683	valid_0's binary_logloss: 0.00873496
[400]	valid_0's avera

M31은 꽤 괜찮음. PR-AUC 자체는 M15/M27보다 낮지만, Recall은 지금까지 제일 좋음.

# Model 32 = M14 + is_online

실험 이유 : is_online은 지금까지 본 금액/반복거래 계열과 다른 거래 채널 정보. 같은 금액·시간·고객 패턴이라도 온라인 거래인지 여부가 추가적인 이상거래 구분력을 주는지 확인하는 실험.

In [23]:
# ============================================================
# Model 32
# M14 + is_online
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m32 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "is_online"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m32]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m32]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("is_online NaN:", df["is_online"].isna().sum())
print("\nis_online value counts:")
print(df["is_online"].value_counts(dropna=False))

model_m32 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m32.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m32 = model_m32.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m32 = (valid_prob_m32 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m32)
roc_auc = roc_auc_score(y_valid, valid_prob_m32)
precision = precision_score(y_valid, valid_pred_m32)
recall = recall_score(y_valid, valid_pred_m32)
f1 = f1_score(y_valid, valid_pred_m32)
cm = confusion_matrix(y_valid, valid_pred_m32)

print("\n===== Model 32 : M14 + is_online =====")
print("Best iteration:", model_m32.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== 주요 후보 대비 =====")
print(f"M15 대비 PR-AUC : {pr_auc - 0.975622:+.6f}")
print(f"M27 대비 PR-AUC : {pr_auc - 0.975753:+.6f}")
print(f"M31 대비 Recall : {recall - 0.891405:+.6f}")
print(f"M15 대비 F1     : {f1 - 0.931633:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
is_online NaN: 0

is_online value counts:
is_online
0    1090393
1     206282
Name: count, dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.955656	valid_0's binary_logloss: 0.0301566
[100]	valid_0's average_precision: 0.964548	valid_0's binary_logloss: 0.0200911
[150]	valid_0's average_precision: 0.968547	valid_0's binary_logloss: 0.0154199
[200]	valid_0's average_precision: 0.970745	valid_0's binary_logloss: 0.0127404
[250]	valid_0's average_precision: 0.971779	valid_0's binary_logloss: 0.0109235
[300]	valid_0's average_precision: 0.972551	valid_0's binary_logloss: 0.00970734
[350]	valid_0's average_precision: 0.973284	valid_0's binary_logloss: 0.00874144
[400]	valid_0's average_precision: 0.973499	valid_0's binary_logloss: 0.00800361
[450]	valid_0's average_precision: 0.973697	valid_0's binary_logloss: 0.0073887
[5

M14보다는 좋아짐. 최상위 후보는 아님. is_online은 유효한 보조 변수.

# Model 33 = M14 + risk_time_22_04

실험 이유: M14에 trans_hour가 이미 있지만, risk_time_22_04는 22~04시라는 특정 야간 위험 구간을 0/1로 명시한 변수. 따라서 LightGBM이 trans_hour만으로 충분히 학습하는지, 아니면 이 수동 위험구간을 별도로 주는 것이 추가 도움이 되는지 확인하는 실험.

In [24]:
# ============================================================
# Model 33
# M14 + risk_time_22_04
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m33 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "risk_time_22_04"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m33]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m33]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("risk_time_22_04 NaN:", df["risk_time_22_04"].isna().sum())
print("\nrisk_time_22_04 value counts:")
print(df["risk_time_22_04"].value_counts(dropna=False))

model_m33 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m33.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m33 = model_m33.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m33 = (valid_prob_m33 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m33)
roc_auc = roc_auc_score(y_valid, valid_prob_m33)
precision = precision_score(y_valid, valid_pred_m33)
recall = recall_score(y_valid, valid_pred_m33)
f1 = f1_score(y_valid, valid_pred_m33)
cm = confusion_matrix(y_valid, valid_pred_m33)

print("\n===== Model 33 : M14 + risk_time_22_04 =====")
print("Best iteration:", model_m33.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== 주요 후보 대비 =====")
print(f"M15 대비 PR-AUC : {pr_auc - 0.975622:+.6f}")
print(f"M27 대비 PR-AUC : {pr_auc - 0.975753:+.6f}")
print(f"M31 대비 Recall : {recall - 0.891405:+.6f}")
print(f"M15 대비 F1     : {f1 - 0.931633:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
risk_time_22_04 NaN: 0

risk_time_22_04 value counts:
risk_time_22_04
0    991793
1    304882
Name: count, dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.937722	valid_0's binary_logloss: 0.029791
[100]	valid_0's average_precision: 0.961461	valid_0's binary_logloss: 0.0201543
[150]	valid_0's average_precision: 0.967778	valid_0's binary_logloss: 0.015408
[200]	valid_0's average_precision: 0.970417	valid_0's binary_logloss: 0.0126972
[250]	valid_0's average_precision: 0.971665	valid_0's binary_logloss: 0.0109127
[300]	valid_0's average_precision: 0.972628	valid_0's binary_logloss: 0.00967765
[350]	valid_0's average_precision: 0.973383	valid_0's binary_logloss: 0.00869016
[400]	valid_0's average_precision: 0.973629	valid_0's binary_logloss: 0.00797683
[450]	valid_0's average_precision: 0.973813	valid_0's binary_logloss

risk_time_22_04는 추가 신호는 있지만 강한 변수는 아니다 정도. trans_hour가 이미 있어서, 22~04시 플래그가 주는 추가 정보가 제한적이었을 가능성이 있음. 다만 이것도 “중복 확정”은 아니고 추가 효과가 크지 않았다고 표현하는 게 정확함.

# Model 34 = M14 + outside_trans_hours_80

In [25]:
# ============================================================
# Model 34
# M14 + outside_trans_hours_80
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m34 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "outside_trans_hours_80"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m34]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m34]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print(
    "outside_trans_hours_80 NaN:",
    df["outside_trans_hours_80"].isna().sum()
)
print("\noutside_trans_hours_80 value counts:")
print(df["outside_trans_hours_80"].value_counts(dropna=False))

model_m34 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m34.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m34 = model_m34.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m34 = (valid_prob_m34 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m34)
roc_auc = roc_auc_score(y_valid, valid_prob_m34)
precision = precision_score(y_valid, valid_pred_m34)
recall = recall_score(y_valid, valid_pred_m34)
f1 = f1_score(y_valid, valid_pred_m34)
cm = confusion_matrix(y_valid, valid_pred_m34)

print("\n===== Model 34 : M14 + outside_trans_hours_80 =====")
print("Best iteration:", model_m34.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== 주요 후보 대비 =====")
print(f"M15 대비 PR-AUC : {pr_auc - 0.975622:+.6f}")
print(f"M27 대비 PR-AUC : {pr_auc - 0.975753:+.6f}")
print(f"M31 대비 Recall : {recall - 0.891405:+.6f}")
print(f"M15 대비 F1     : {f1 - 0.931633:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
outside_trans_hours_80 NaN: 0

outside_trans_hours_80 value counts:
outside_trans_hours_80
0    1028541
1     268134
Name: count, dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.946591	valid_0's binary_logloss: 0.0298747
[100]	valid_0's average_precision: 0.959464	valid_0's binary_logloss: 0.0195396
[150]	valid_0's average_precision: 0.966479	valid_0's binary_logloss: 0.0150387
[200]	valid_0's average_precision: 0.969138	valid_0's binary_logloss: 0.0124861
[250]	valid_0's average_precision: 0.970473	valid_0's binary_logloss: 0.0106892
[300]	valid_0's average_precision: 0.971493	valid_0's binary_logloss: 0.00949865
[350]	valid_0's average_precision: 0.972532	valid_0's binary_logloss: 0.00852402
[400]	valid_0's average_precision: 0.972891	valid_0's binary_logloss: 0.00780731
[450]	valid_0's average_precision: 0.973293

M34는 M14보다 소폭 개선, 최상위 후보는 아님.

outside_trans_hours_80은 개인별 시간 이탈 신호가 약간 도움은 됐지만 강한 변수는 아니다 정도.

# Model 35 = M14 + speed_2

실험 이유 : speed_2는 이전 거래와 현재 거래 사이의 이동속도 원값. 거래 장소 이동이 비정상적으로 빠른 경우가 이상거래 신호가 되는지 확인.

In [26]:
# ============================================================
# Model 35
# M14 + speed_2
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m35 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "speed_2"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m35]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m35]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("speed_2 NaN:", df["speed_2"].isna().sum())
print("\nspeed_2 분포:")
print(df["speed_2"].describe())

model_m35 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m35.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m35 = model_m35.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m35 = (valid_prob_m35 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m35)
roc_auc = roc_auc_score(y_valid, valid_prob_m35)
precision = precision_score(y_valid, valid_pred_m35)
recall = recall_score(y_valid, valid_pred_m35)
f1 = f1_score(y_valid, valid_pred_m35)
cm = confusion_matrix(y_valid, valid_pred_m35)

print("\n===== Model 35 : M14 + speed_2 =====")
print("Best iteration:", model_m35.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== 주요 후보 대비 =====")
print(f"M15 대비 PR-AUC : {pr_auc - 0.975622:+.6f}")
print(f"M27 대비 PR-AUC : {pr_auc - 0.975753:+.6f}")
print(f"M31 대비 Recall : {recall - 0.891405:+.6f}")
print(f"M15 대비 F1     : {f1 - 0.931633:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
speed_2 NaN: 0

speed_2 분포:
count    1.296675e+06
mean     8.605355e+01
std      4.590036e+02
min      0.000000e+00
25%      2.968340e+00
50%      1.162299e+01
75%      3.933591e+01
max      1.582018e+04
Name: speed_2, dtype: float64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.936093	valid_0's binary_logloss: 0.0294912
[100]	valid_0's average_precision: 0.955615	valid_0's binary_logloss: 0.018747
[150]	valid_0's average_precision: 0.965373	valid_0's binary_logloss: 0.0140686
[200]	valid_0's average_precision: 0.969319	valid_0's binary_logloss: 0.0114821
[250]	valid_0's average_precision: 0.97055	valid_0's binary_logloss: 0.00974334
[300]	valid_0's average_precision: 0.971555	valid_0's binary_logloss: 0.00861414
[350]	valid_0's average_precision: 0.972553	valid_0's binary_logloss: 0.00776155
[400]	valid_0's average_precision:

M35는 M14보다 거의 비슷하거나 약간 아쉬운 결과.

# Model 36 = M14 + high_speed

high_speed는 100km/h 이상 여부를 0/1로 단순화한 변수. 그래서 원값 전체를 주는 것보다, 비정상적으로 빠른 이동만 강조한 플래그가 더 유용한지 비교.

In [27]:
# ============================================================
# Model 36
# M14 + high_speed
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m36 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "high_speed"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m36]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m36]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("high_speed NaN:", df["high_speed"].isna().sum())
print("\nhigh_speed value counts:")
print(df["high_speed"].value_counts(dropna=False))

model_m36 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m36.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m36 = model_m36.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m36 = (valid_prob_m36 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m36)
roc_auc = roc_auc_score(y_valid, valid_prob_m36)
precision = precision_score(y_valid, valid_pred_m36)
recall = recall_score(y_valid, valid_pred_m36)
f1 = f1_score(y_valid, valid_pred_m36)
cm = confusion_matrix(y_valid, valid_pred_m36)

print("\n===== Model 36 : M14 + high_speed =====")
print("Best iteration:", model_m36.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== M35(speed_2) 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973496:+.6f}")
print(f"Precision : {precision - 0.971863:+.6f}")
print(f"Recall    : {recall - 0.883438:+.6f}")
print(f"F1-score  : {f1 - 0.925544:+.6f}")

print("\n===== 주요 후보 대비 =====")
print(f"M15 대비 PR-AUC : {pr_auc - 0.975622:+.6f}")
print(f"M27 대비 PR-AUC : {pr_auc - 0.975753:+.6f}")
print(f"M31 대비 Recall : {recall - 0.891405:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
high_speed NaN: 0

high_speed value counts:
high_speed
0    1139777
1     156898
Name: count, dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.952055	valid_0's binary_logloss: 0.0290365
[100]	valid_0's average_precision: 0.963881	valid_0's binary_logloss: 0.0194688
[150]	valid_0's average_precision: 0.968462	valid_0's binary_logloss: 0.0148493
[200]	valid_0's average_precision: 0.971135	valid_0's binary_logloss: 0.0122481
[250]	valid_0's average_precision: 0.972472	valid_0's binary_logloss: 0.0104582
[300]	valid_0's average_precision: 0.97318	valid_0's binary_logloss: 0.00931945
[350]	valid_0's average_precision: 0.973793	valid_0's binary_logloss: 0.00838744
[400]	valid_0's average_precision: 0.974052	valid_0's binary_logloss: 0.00771774
[450]	valid_0's average_precision: 0.974414	valid_0's binary_logloss: 0.0071064


M36은 내 예상과 다른 결과가 나옴. M35보다 확실히 낫고, M14 대비도 꽤 괜찮게 개선됨.

speed_2보다 high_speed가 전 지표에서 더 좋았음. 즉 이 실험에서는 속도 원값 전체보다 “100km/h 이상인가”라는 임계값 플래그가 더 유용했다고 말할 수 있음. 다만 이건 현재 고정 파라미터/고정 split에서의 결과.

# Model 37 = M14 + is_high_amt

In [28]:
# ============================================================
# Model 37
# M14 + is_high_amt
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m37 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "is_high_amt"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m37]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m37]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("is_high_amt NaN:", df["is_high_amt"].isna().sum())
print("\nis_high_amt value counts:")
print(df["is_high_amt"].value_counts(dropna=False))

model_m37 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m37.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m37 = model_m37.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m37 = (valid_prob_m37 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m37)
roc_auc = roc_auc_score(y_valid, valid_prob_m37)
precision = precision_score(y_valid, valid_pred_m37)
recall = recall_score(y_valid, valid_pred_m37)
f1 = f1_score(y_valid, valid_pred_m37)
cm = confusion_matrix(y_valid, valid_pred_m37)

print("\n===== Model 37 : M14 + is_high_amt =====")
print("Best iteration:", model_m37.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== 주요 후보 대비 =====")
print(f"M15 대비 PR-AUC : {pr_auc - 0.975622:+.6f}")
print(f"M27 대비 PR-AUC : {pr_auc - 0.975753:+.6f}")
print(f"M36 대비 PR-AUC : {pr_auc - 0.975002:+.6f}")
print(f"M31 대비 Recall : {recall - 0.891405:+.6f}")

Train: (907672, 8)
Valid: (389003, 8)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
is_high_amt NaN: 0

is_high_amt value counts:
is_high_amt
0    1281044
1      15631
Name: count, dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.951381	valid_0's binary_logloss: 0.0295821
[100]	valid_0's average_precision: 0.964156	valid_0's binary_logloss: 0.0195094
[150]	valid_0's average_precision: 0.968705	valid_0's binary_logloss: 0.0149783
[200]	valid_0's average_precision: 0.970977	valid_0's binary_logloss: 0.0124741
[250]	valid_0's average_precision: 0.972125	valid_0's binary_logloss: 0.0106632
[300]	valid_0's average_precision: 0.973217	valid_0's binary_logloss: 0.00944486
[350]	valid_0's average_precision: 0.973681	valid_0's binary_logloss: 0.0085434
[400]	valid_0's average_precision: 0.974157	valid_0's binary_logloss: 0.00779805
[450]	valid_0's average_precision: 0.974385	valid_0's binary_logloss: 0.00719

M37 꽤 좋음. M14보다 명확히 개선됐고, PR-AUC 기준으로는 지금 상위권.

특히 is_high_amt는 amt가 이미 있는데도 추가 이득이 있었음. 그래서 지금 실험에서는 “500 이상”이라는 명시적 고액거래 경계가 LightGBM에 추가 신호를 줬다고 볼 수 있음. 다만 여전히 M15/M27보다는 PR-AUC가 조금 낮음.

# Model 38 = M14 + category_recent_fraud_rate + category_recent_fraud_rate_missing

실험 이유: 

- category_recent_fraud_rate는 업종별 최근 7일 이상거래율
- category_recent_fraud_rate_missing은 그 이력이 충분하지 않아 값을 신뢰하기 어려운 상태를 표시. 

그래서 두 변수를 같이 넣었을 때, 단순히 업종 위험도만 주는 것보다 “그 위험도가 실제 이력 기반인지 여부”까지 알려주는 것이 추가 도움이 되는지 확인하는 실험.

In [29]:
# ============================================================
# Model 38
# M14 + category_recent_fraud_rate
#     + category_recent_fraud_rate_missing
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m38 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "category_recent_fraud_rate",
    "category_recent_fraud_rate_missing"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m38]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m38]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print(
    "category_recent_fraud_rate NaN:",
    df["category_recent_fraud_rate"].isna().sum()
)
print(
    "category_recent_fraud_rate_missing NaN:",
    df["category_recent_fraud_rate_missing"].isna().sum()
)

print("\ncategory_recent_fraud_rate_missing value counts:")
print(
    df["category_recent_fraud_rate_missing"]
    .value_counts(dropna=False)
)

model_m38 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m38.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m38 = model_m38.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m38 = (valid_prob_m38 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m38)
roc_auc = roc_auc_score(y_valid, valid_prob_m38)
precision = precision_score(y_valid, valid_pred_m38)
recall = recall_score(y_valid, valid_pred_m38)
f1 = f1_score(y_valid, valid_pred_m38)
cm = confusion_matrix(y_valid, valid_pred_m38)

print("\n===== Model 38 =====")
print("Best iteration:", model_m38.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== M22 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973320:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999321:+.6f}")
print(f"Precision : {precision - 0.975383:+.6f}")
print(f"Recall    : {recall - 0.880503:+.6f}")
print(f"F1-score  : {f1 - 0.925518:+.6f}")

print("\n===== 주요 후보 대비 =====")
print(f"M15 대비 PR-AUC : {pr_auc - 0.975622:+.6f}")
print(f"M27 대비 PR-AUC : {pr_auc - 0.975753:+.6f}")
print(f"M37 대비 PR-AUC : {pr_auc - 0.975437:+.6f}")

Train: (907672, 9)
Valid: (389003, 9)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797
category_recent_fraud_rate NaN: 0
category_recent_fraud_rate_missing NaN: 0

category_recent_fraud_rate_missing value counts:
category_recent_fraud_rate_missing
0    1296661
1         14
Name: count, dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.95406	valid_0's binary_logloss: 0.030569
[100]	valid_0's average_precision: 0.963627	valid_0's binary_logloss: 0.0202929
[150]	valid_0's average_precision: 0.967187	valid_0's binary_logloss: 0.015606
[200]	valid_0's average_precision: 0.969313	valid_0's binary_logloss: 0.0128682
[250]	valid_0's average_precision: 0.970361	valid_0's binary_logloss: 0.0109275
[300]	valid_0's average_precision: 0.971387	valid_0's binary_logloss: 0.00957965
[350]	valid_0's average_precision: 0.972037	valid_0's binary_logloss: 0.00853996
[400]	valid_0's average_precision: 0.972423	valid_0's bin

예상대로 나옴.

category_recent_fraud_rate_missing=1이 전체 1,296,675건 중 14건뿐이라서, 사실상 모델이 이 변수를 활용할 기회가 거의 없었음. 그래서 M38 결과가 M22와 완전히 동일하게 나온 것도 자연스러움.

즉,

- category_recent_fraud_rate_missing은 이 데이터에서는 너무 희소함
- category_recent_fraud_rate에 missing 여부를 추가해도 성능 변화가 없음
- 따라서 이 변수는 최종 후보에서 우선 제외해도 무방


Model 39 = M14 + 여러 유효 변수 8개 동시 추가

'서로 다른' 정보축을 한 번에 넣어서 개별적으로는 약하거나 중간 정도였던 변수들이 함께 있을 때 상호보완적으로 성능이 올라가는지 확인하는 단계.

- amt_zscore_card → 고객 기준 금액 이상성
- prior_normal_median_amt → 과거 정상거래 기준 금액
- count_30min → 단기 거래 빈도
- merchant_change_count → 가맹점 변화
- is_online → 거래 채널
- high_speed → 비정상 이동속도 플래그
- is_high_amt → 고액거래 플래그
- has_prior_normal_transaction → 과거 정상거래 이력 존재 여부

In [30]:
# ============================================================
# Model 39
# M14 + 다변수 확장 조합
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m39 = [
    # M14
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",

    # 추가 변수
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",
    "merchant_change_count",
    "is_online",
    "high_speed",
    "is_high_amt",
    "has_prior_normal_transaction"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m39]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m39]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Number of features:", len(features_m39))
print("Features:")
for f in features_m39:
    print("-", f)

print("\nTrain:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("\n===== NaN count =====")
print(df[features_m39].isna().sum())

model_m39 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m39.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m39 = model_m39.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m39 = (valid_prob_m39 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m39)
roc_auc = roc_auc_score(y_valid, valid_prob_m39)
precision = precision_score(y_valid, valid_pred_m39)
recall = recall_score(y_valid, valid_pred_m39)
f1 = f1_score(y_valid, valid_pred_m39)
cm = confusion_matrix(y_valid, valid_pred_m39)

print("\n===== Model 39 : Expanded Multi-feature Model =====")
print("Best iteration:", model_m39.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== 주요 후보 대비 =====")
print(f"M15 대비 PR-AUC : {pr_auc - 0.975622:+.6f}")
print(f"M27 대비 PR-AUC : {pr_auc - 0.975753:+.6f}")
print(f"M31 대비 Recall : {recall - 0.891405:+.6f}")
print(f"M15 대비 F1     : {f1 - 0.931633:+.6f}")
print(f"M37 대비 PR-AUC : {pr_auc - 0.975437:+.6f}")

Number of features: 15
Features:
- category
- amt
- trans_hour
- age
- amt_to_prior_median_ratio
- rolling_sum_amt_1h
- recent_24h_high_amt_count
- amt_zscore_card
- prior_normal_median_amt
- count_30min
- merchant_change_count
- is_online
- high_speed
- is_high_amt
- has_prior_normal_transaction

Train: (907672, 15)
Valid: (389003, 15)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797

===== NaN count =====
category                           0
amt                                0
trans_hour                         0
age                                0
amt_to_prior_median_ratio       1649
rolling_sum_amt_1h                 0
recent_24h_high_amt_count          0
amt_zscore_card                    0
prior_normal_median_amt         1649
count_30min                        0
merchant_change_count              0
is_online                          0
high_speed                         0
is_high_amt                        0
has_prior_normal_transaction       0
dtype: int

- PR-AUC는 M14보다 오히려 하락: 0.973016
- Precision은 크게 상승: 0.985521
- F1은 지금까지 꽤 높은 편: 0.932391
- 하지만 Recall은 0.884696으로 M31보다 낮음.

즉 변수 15개를 넣었다고 전체적으로 더 좋아진 건 아니고, 고정 임계값에서는 FP를 많이 줄이면서 Precision/F1 쪽으로 성향이 바뀐 모델.

# Model 40 = M14 + amt_zscore_card + prior_normal_median_amt + count_30min

서로 역할이 다른 3개를 묶음

- amt_zscore_card → 고객 기준 금액 이상성
- prior_normal_median_amt → 과거 정상거래의 기준 금액 수준
- count_30min → 단기 거래 빈도

즉 금액 이상 + 정상 기준 + 단기 반복 패턴을 함께 썼을 때 M14보다 더 안정적으로 좋아지는지 확인하는 단계.

In [31]:
# ============================================================
# Model 40
# M14 + amt_zscore_card
#     + prior_normal_median_amt
#     + count_30min
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m40 = [
    # M14
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",

    # 추가 변수
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m40]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m40]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Number of features:", len(features_m40))
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("\n===== NaN count =====")
print(df[features_m40].isna().sum())

model_m40 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m40.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m40 = model_m40.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m40 = (valid_prob_m40 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m40)
roc_auc = roc_auc_score(y_valid, valid_prob_m40)
precision = precision_score(y_valid, valid_pred_m40)
recall = recall_score(y_valid, valid_pred_m40)
f1 = f1_score(y_valid, valid_pred_m40)
cm = confusion_matrix(y_valid, valid_pred_m40)

print("\n===== Model 40 =====")
print("Best iteration:", model_m40.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M14 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.973656:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999451:+.6f}")
print(f"Precision : {precision - 0.969613:+.6f}")
print(f"Recall    : {recall - 0.883019:+.6f}")
print(f"F1-score  : {f1 - 0.924292:+.6f}")

print("\n===== 주요 후보 대비 =====")
print(f"M15 대비 PR-AUC : {pr_auc - 0.975622:+.6f}")
print(f"M27 대비 PR-AUC : {pr_auc - 0.975753:+.6f}")
print(f"M31 대비 Recall : {recall - 0.891405:+.6f}")
print(f"M39 대비 PR-AUC : {pr_auc - 0.973016:+.6f}")
print(f"M39 대비 F1     : {f1 - 0.932391:+.6f}")

Number of features: 10
Train: (907672, 10)
Valid: (389003, 10)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797

===== NaN count =====
category                        0
amt                             0
trans_hour                      0
age                             0
amt_to_prior_median_ratio    1649
rolling_sum_amt_1h              0
recent_24h_high_amt_count       0
amt_zscore_card                 0
prior_normal_median_amt      1649
count_30min                     0
dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.949551	valid_0's binary_logloss: 0.0280795
[100]	valid_0's average_precision: 0.964812	valid_0's binary_logloss: 0.0169998
[150]	valid_0's average_precision: 0.96935	valid_0's binary_logloss: 0.0123124
[200]	valid_0's average_precision: 0.971538	valid_0's binary_logloss: 0.00967927
[250]	valid_0's average_precision: 0.972752	valid_0's binary_logloss: 0.00789298
[300]	valid_0's average_pre

M40은 지금까지 결과 중 가장 균형이 좋아 보임.

- PR-AUC 0.975945 → 지금까지 최고
- Precision 0.989232 → 매우 높음
- Recall 0.885954 → M31보다는 낮지만 크게 뒤처지진 않음
- F1 0.934749 → 지금까지 최고

그리고 M39보다도

- PR-AUC +0.002929
- F1 +0.002358

이라서 변수를 무작정 많이 넣는 것보다, 서로 역할이 다른 강한 변수 3개를 추가한 쪽이 훨씬 나음.

# Model 41 = 'M40' + merchant_change_count + is_online

- M40은 금액/정상기준/단기빈도 중심
- 여기에 merchant_change_count로 가맹점 변화 행동
- is_online으로 거래 채널

을 추가해서 행동·채널 정보가 M40에 추가 보완 효과를 주는가?

In [32]:
# ============================================================
# Model 41
# M40 + merchant_change_count + is_online
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m41 = [
    # M14
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",

    # M40 추가 변수
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",

    # M41 추가 변수
    "merchant_change_count",
    "is_online"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m41]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m41]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Number of features:", len(features_m41))
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("\n===== NaN count =====")
print(df[features_m41].isna().sum())

model_m41 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m41.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m41 = model_m41.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m41 = (valid_prob_m41 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m41)
roc_auc = roc_auc_score(y_valid, valid_prob_m41)
precision = precision_score(y_valid, valid_pred_m41)
recall = recall_score(y_valid, valid_pred_m41)
f1 = f1_score(y_valid, valid_pred_m41)
cm = confusion_matrix(y_valid, valid_pred_m41)

print("\n===== Model 41 =====")
print("Best iteration:", model_m41.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M40 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975945:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999375:+.6f}")
print(f"Precision : {precision - 0.989232:+.6f}")
print(f"Recall    : {recall - 0.885954:+.6f}")
print(f"F1-score  : {f1 - 0.934749:+.6f}")

print("\n===== 기존 주요 후보 대비 =====")
print(f"M15 대비 PR-AUC : {pr_auc - 0.975622:+.6f}")
print(f"M27 대비 PR-AUC : {pr_auc - 0.975753:+.6f}")
print(f"M31 대비 Recall : {recall - 0.891405:+.6f}")
print(f"M39 대비 PR-AUC : {pr_auc - 0.973016:+.6f}")

Number of features: 12
Train: (907672, 12)
Valid: (389003, 12)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797

===== NaN count =====
category                        0
amt                             0
trans_hour                      0
age                             0
amt_to_prior_median_ratio    1649
rolling_sum_amt_1h              0
recent_24h_high_amt_count       0
amt_zscore_card                 0
prior_normal_median_amt      1649
count_30min                     0
merchant_change_count           0
is_online                       0
dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.929863	valid_0's binary_logloss: 0.0294109
[100]	valid_0's average_precision: 0.951747	valid_0's binary_logloss: 0.0174095
[150]	valid_0's average_precision: 0.963525	valid_0's binary_logloss: 0.0126031
[200]	valid_0's average_precision: 0.967259	valid_0's binary_logloss: 0.0100584
[250]	valid_0's average_precision: 0.969

M41은 M40보다 명확히 나빠짐. 그래서 merchant_change_count + is_online을 같이 추가하는 방향은 현재 기준에선 별로.

현재까지는 M40이 더 좋은 후보.

두 변수를 하나씩 추가해서 보기.

# Model 42 = M40 + merchant_change_count

M41에서 merchant_change_count + is_online을 같이 넣었더니 M40보다 성능이 떨어졌음. 그래서 두 변수 중 어떤 쪽이 영향을 줬는지 분리해서 확인하는 단계.

merchant_change_count는 직전 거래 대비 가맹점이 바뀌었는지를 나타내는 행동 변화 변수라서, M40의 금액/빈도 중심 정보에 가맹점 이동 행동이 추가 도움이 되는지 보는 실험.

In [33]:
# ============================================================
# Model 42
# M40 + merchant_change_count
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m42 = [
    # M14
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",

    # M40 추가 변수
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",

    # M42 추가 변수
    "merchant_change_count"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m42]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m42]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Number of features:", len(features_m42))
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("\n===== NaN count =====")
print(df[features_m42].isna().sum())

model_m42 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m42.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m42 = model_m42.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m42 = (valid_prob_m42 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m42)
roc_auc = roc_auc_score(y_valid, valid_prob_m42)
precision = precision_score(y_valid, valid_pred_m42)
recall = recall_score(y_valid, valid_pred_m42)
f1 = f1_score(y_valid, valid_pred_m42)
cm = confusion_matrix(y_valid, valid_pred_m42)

print("\n===== Model 42 =====")
print("Best iteration:", model_m42.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M40 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975945:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999375:+.6f}")
print(f"Precision : {precision - 0.989232:+.6f}")
print(f"Recall    : {recall - 0.885954:+.6f}")
print(f"F1-score  : {f1 - 0.934749:+.6f}")

print("\n===== M41 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.974929:+.6f}")
print(f"Precision : {precision - 0.985935:+.6f}")
print(f"Recall    : {recall - 0.881761:+.6f}")
print(f"F1-score  : {f1 - 0.930943:+.6f}")

Number of features: 11
Train: (907672, 11)
Valid: (389003, 11)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797

===== NaN count =====
category                        0
amt                             0
trans_hour                      0
age                             0
amt_to_prior_median_ratio    1649
rolling_sum_amt_1h              0
recent_24h_high_amt_count       0
amt_zscore_card                 0
prior_normal_median_amt      1649
count_30min                     0
merchant_change_count           0
dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.932323	valid_0's binary_logloss: 0.029065
[100]	valid_0's average_precision: 0.958278	valid_0's binary_logloss: 0.0174701
[150]	valid_0's average_precision: 0.967036	valid_0's binary_logloss: 0.0124577
[200]	valid_0's average_precision: 0.970336	valid_0's binary_logloss: 0.00983928
[250]	valid_0's average_precision: 0.972612	valid_0's binary_logloss: 0.00

M42는 꽤 의미 있음. PR-AUC는 지금까지 최고로 올라갔는데, Recall/F1은 M40보다 떨어짐.

M42가 M41보다 좋으니까, M41 하락에는 is_online이 같이 들어간 영향이 있었을 가능성이 커짐. 물론 이것도 M43로 직접 확인해야 확실함.

# Model 43 = M40 + is_online

is_online만 따로 넣어서, M41 성능 하락에 is_online이 영향을 줬는지 분리해서 확인하는 실험

In [34]:
# ============================================================
# Model 43
# M40 + is_online
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m43 = [
    # M14
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",

    # M40 추가 변수
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",

    # M43 추가 변수
    "is_online"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m43]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m43]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Number of features:", len(features_m43))
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("\n===== NaN count =====")
print(df[features_m43].isna().sum())

model_m43 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m43.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m43 = model_m43.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m43 = (valid_prob_m43 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m43)
roc_auc = roc_auc_score(y_valid, valid_prob_m43)
precision = precision_score(y_valid, valid_pred_m43)
recall = recall_score(y_valid, valid_pred_m43)
f1 = f1_score(y_valid, valid_pred_m43)
cm = confusion_matrix(y_valid, valid_pred_m43)

print("\n===== Model 43 =====")
print("Best iteration:", model_m43.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M40 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975945:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999375:+.6f}")
print(f"Precision : {precision - 0.989232:+.6f}")
print(f"Recall    : {recall - 0.885954:+.6f}")
print(f"F1-score  : {f1 - 0.934749:+.6f}")

print("\n===== M42 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.976749:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999448:+.6f}")
print(f"Precision : {precision - 0.988257:+.6f}")
print(f"Recall    : {recall - 0.882180:+.6f}")
print(f"F1-score  : {f1 - 0.932211:+.6f}")

print("\n===== M41 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.974929:+.6f}")
print(f"Precision : {precision - 0.985935:+.6f}")
print(f"Recall    : {recall - 0.881761:+.6f}")
print(f"F1-score  : {f1 - 0.930943:+.6f}")

Number of features: 11
Train: (907672, 11)
Valid: (389003, 11)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797

===== NaN count =====
category                        0
amt                             0
trans_hour                      0
age                             0
amt_to_prior_median_ratio    1649
rolling_sum_amt_1h              0
recent_24h_high_amt_count       0
amt_zscore_card                 0
prior_normal_median_amt      1649
count_30min                     0
is_online                       0
dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.932162	valid_0's binary_logloss: 0.0291015
[100]	valid_0's average_precision: 0.957932	valid_0's binary_logloss: 0.0173497
[150]	valid_0's average_precision: 0.966591	valid_0's binary_logloss: 0.0124911
[200]	valid_0's average_precision: 0.970158	valid_0's binary_logloss: 0.00989202
[250]	valid_0's average_precision: 0.972175	valid_0's binary_logloss: 0.0

M43 결과까지 보면 꽤 명확해짐.

- is_online을 M40에 단독 추가하니 PR-AUC, Precision, Recall, F1이 전반적으로 하락
- 반면 merchant_change_count 단독 추가한 M42는 PR-AUC 최고치 0.976749까지 올라감

그래서 현재 조합에서는 is_online이 도움이 되지 않았고, M41이 떨어진 데에는 is_online 영향이 컸을 가능성이 높아짐.

다만, “원인 확정”보다는 현재 split/파라미터에서 그렇게 관찰됐다고 보는 게 정확함.

# Model 14 = M40 + high_speed

실험 이유 : M40은 금액/정상거래 기준/단기 빈도 중심이고, high_speed는 이동 이상성이라는 다른 정보축. M36에서 M14에 단독 추가했을 때 PR-AUC가 0.975002까지 올라갔으니까, 이번에는 강한 M40 조합 위에서도 이동 이상 플래그가 추가 도움이 되는지 확인.

In [35]:
# ============================================================
# Model 44
# M40 + high_speed
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m44 = [
    # M14
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",

    # M40 추가 변수
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",

    # M44 추가 변수
    "high_speed"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m44]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m44]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Number of features:", len(features_m44))
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("\n===== NaN count =====")
print(df[features_m44].isna().sum())

model_m44 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m44.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m44 = model_m44.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m44 = (valid_prob_m44 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m44)
roc_auc = roc_auc_score(y_valid, valid_prob_m44)
precision = precision_score(y_valid, valid_pred_m44)
recall = recall_score(y_valid, valid_pred_m44)
f1 = f1_score(y_valid, valid_pred_m44)
cm = confusion_matrix(y_valid, valid_pred_m44)

print("\n===== Model 44 =====")
print("Best iteration:", model_m44.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M40 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975945:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999375:+.6f}")
print(f"Precision : {precision - 0.989232:+.6f}")
print(f"Recall    : {recall - 0.885954:+.6f}")
print(f"F1-score  : {f1 - 0.934749:+.6f}")

print("\n===== M42 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.976749:+.6f}")
print(f"Precision : {precision - 0.988257:+.6f}")
print(f"Recall    : {recall - 0.882180:+.6f}")
print(f"F1-score  : {f1 - 0.932211:+.6f}")

print("\n===== M36 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975002:+.6f}")
print(f"Recall    : {recall - 0.887212:+.6f}")
print(f"F1-score  : {f1 - 0.928885:+.6f}")

Number of features: 11
Train: (907672, 11)
Valid: (389003, 11)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797

===== NaN count =====
category                        0
amt                             0
trans_hour                      0
age                             0
amt_to_prior_median_ratio    1649
rolling_sum_amt_1h              0
recent_24h_high_amt_count       0
amt_zscore_card                 0
prior_normal_median_amt      1649
count_30min                     0
high_speed                      0
dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.929974	valid_0's binary_logloss: 0.0296178
[100]	valid_0's average_precision: 0.95839	valid_0's binary_logloss: 0.0174819
[150]	valid_0's average_precision: 0.965843	valid_0's binary_logloss: 0.0126489
[200]	valid_0's average_precision: 0.97076	valid_0's binary_logloss: 0.00992695
[250]	valid_0's average_precision: 0.972683	valid_0's binary_logloss: 0.008

현재 성격은 

- M42: PR-AUC 최고 0.976749
- M40: F1 최고 0.934749, Precision도 매우 높음
- M44: PR-AUC는 M42에 가깝고 Recall/F1은 M42보다 조금 좋음

# Model 45 = M40 + is_high_amt

M40은 이미 금액 관련 정보가 강하게 들어가 있지만, is_high_amt는 500 이상 고액거래 여부를 명시적으로 알려주는 플래그.

강한 M40 조합 위에서도 이 임계값 정보가 추가 이득을 주는지 확인.

In [36]:
# ============================================================
# Model 45
# M40 + is_high_amt
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m45 = [
    # M14
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",

    # M40 추가 변수
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",

    # M45 추가 변수
    "is_high_amt"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m45]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m45]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Number of features:", len(features_m45))
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("\n===== NaN count =====")
print(df[features_m45].isna().sum())

model_m45 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m45.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m45 = model_m45.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m45 = (valid_prob_m45 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m45)
roc_auc = roc_auc_score(y_valid, valid_prob_m45)
precision = precision_score(y_valid, valid_pred_m45)
recall = recall_score(y_valid, valid_pred_m45)
f1 = f1_score(y_valid, valid_pred_m45)
cm = confusion_matrix(y_valid, valid_pred_m45)

print("\n===== Model 45 =====")
print("Best iteration:", model_m45.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M40 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975945:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999375:+.6f}")
print(f"Precision : {precision - 0.989232:+.6f}")
print(f"Recall    : {recall - 0.885954:+.6f}")
print(f"F1-score  : {f1 - 0.934749:+.6f}")

print("\n===== M42 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.976749:+.6f}")
print(f"Precision : {precision - 0.988257:+.6f}")
print(f"Recall    : {recall - 0.882180:+.6f}")
print(f"F1-score  : {f1 - 0.932211:+.6f}")

print("\n===== M44 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.976510:+.6f}")
print(f"Precision : {precision - 0.985528:+.6f}")
print(f"Recall    : {recall - 0.885115:+.6f}")
print(f"F1-score  : {f1 - 0.932626:+.6f}")

print("\n===== M37 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975437:+.6f}")
print(f"Recall    : {recall - 0.887212:+.6f}")
print(f"F1-score  : {f1 - 0.928885:+.6f}")

Number of features: 11
Train: (907672, 11)
Valid: (389003, 11)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797

===== NaN count =====
category                        0
amt                             0
trans_hour                      0
age                             0
amt_to_prior_median_ratio    1649
rolling_sum_amt_1h              0
recent_24h_high_amt_count       0
amt_zscore_card                 0
prior_normal_median_amt      1649
count_30min                     0
is_high_amt                     0
dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.934485	valid_0's binary_logloss: 0.0290728
[100]	valid_0's average_precision: 0.959373	valid_0's binary_logloss: 0.0173469
[150]	valid_0's average_precision: 0.966427	valid_0's binary_logloss: 0.0125269
[200]	valid_0's average_precision: 0.970553	valid_0's binary_logloss: 0.00984377
[250]	valid_0's average_precision: 0.972442	valid_0's binary_logloss: 0.0

is_high_amt는 랭킹 성능(PR-AUC)은 아주 조금 올렸지만, 고정 threshold 기준 탐지 성능은 M40보다 떨어짐. 그래서 M42나 M44처럼 “PR-AUC 개선형”에 가까움.

# Model 46 = M40 + high_speed + is_high_amt

실험 이유 : 
M44에서 high_speed, M45에서 is_high_amt를 각각 M40에 추가했을 때 둘 다 PR-AUC는 소폭 상승. 그래서 이번에는 두 플래그를 같이 넣어서, 이동 이상 + 고액거래 여부가 함께 있을 때 추가적인 보완 효과가 있는지 확인

In [37]:
# ============================================================
# Model 46
# M40 + high_speed + is_high_amt
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m46 = [
    # M14
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",

    # M40 추가 변수
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",

    # M46 추가 변수
    "high_speed",
    "is_high_amt"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m46]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m46]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Number of features:", len(features_m46))
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("\n===== NaN count =====")
print(df[features_m46].isna().sum())

model_m46 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m46.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m46 = model_m46.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m46 = (valid_prob_m46 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m46)
roc_auc = roc_auc_score(y_valid, valid_prob_m46)
precision = precision_score(y_valid, valid_pred_m46)
recall = recall_score(y_valid, valid_pred_m46)
f1 = f1_score(y_valid, valid_pred_m46)
cm = confusion_matrix(y_valid, valid_pred_m46)

print("\n===== Model 46 =====")
print("Best iteration:", model_m46.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M40 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975945:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999375:+.6f}")
print(f"Precision : {precision - 0.989232:+.6f}")
print(f"Recall    : {recall - 0.885954:+.6f}")
print(f"F1-score  : {f1 - 0.934749:+.6f}")

print("\n===== M44 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.976510:+.6f}")
print(f"Precision : {precision - 0.985528:+.6f}")
print(f"Recall    : {recall - 0.885115:+.6f}")
print(f"F1-score  : {f1 - 0.932626:+.6f}")

print("\n===== M45 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.976175:+.6f}")
print(f"Precision : {precision - 0.987816:+.6f}")
print(f"Recall    : {recall - 0.883857:+.6f}")
print(f"F1-score  : {f1 - 0.932950:+.6f}")

print("\n===== 현재 PR-AUC 최고 M42 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.976749:+.6f}")
print(f"Precision : {precision - 0.988257:+.6f}")
print(f"Recall    : {recall - 0.882180:+.6f}")
print(f"F1-score  : {f1 - 0.932211:+.6f}")

Number of features: 12
Train: (907672, 12)
Valid: (389003, 12)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797

===== NaN count =====
category                        0
amt                             0
trans_hour                      0
age                             0
amt_to_prior_median_ratio    1649
rolling_sum_amt_1h              0
recent_24h_high_amt_count       0
amt_zscore_card                 0
prior_normal_median_amt      1649
count_30min                     0
high_speed                      0
is_high_amt                     0
dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.920277	valid_0's binary_logloss: 0.0295187
[100]	valid_0's average_precision: 0.946164	valid_0's binary_logloss: 0.0173723
[150]	valid_0's average_precision: 0.960767	valid_0's binary_logloss: 0.0125774
[200]	valid_0's average_precision: 0.965766	valid_0's binary_logloss: 0.00995801
[250]	valid_0's average_precision: 0.96

M46은 둘을 같이 넣는 조합은 별로임. high_speed와 is_high_amt를 각각 넣었을 때는 PR-AUC가 올랐는데, 같이 넣으니까 오히려 크게 떨어짐.

현재 split/파라미터/column sampling 조건에서 동시 투입 시 추가 이득이 없고 오히려 성능이 하락했다.

# Model 47 = M40 + has_prior_normal_transaction

실험 이유 : M40에는 amt_to_prior_median_ratio와 prior_normal_median_amt가 이미 들어가 있는데, 둘 다 과거 정상거래가 없는 고객 구간에서는 NaN이 생김.
has_prior_normal_transaction은 그 상태를 0/1로 명시해주니까, LightGBM이 단순히 NaN 자체를 처리하는 것보다 “정상거래 이력이 존재하는가”라는 상태 정보를 따로 주는 게 도움이 되는지 확인하는 실험.

In [38]:
# ============================================================
# Model 47
# M40 + has_prior_normal_transaction
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m47 = [
    # M14
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",

    # M40 추가 변수
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",

    # M47 추가 변수
    "has_prior_normal_transaction"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m47]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m47]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Number of features:", len(features_m47))
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("\n===== NaN count =====")
print(df[features_m47].isna().sum())

print("\nhas_prior_normal_transaction value counts:")
print(df["has_prior_normal_transaction"].value_counts(dropna=False))

model_m47 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m47.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m47 = model_m47.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m47 = (valid_prob_m47 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m47)
roc_auc = roc_auc_score(y_valid, valid_prob_m47)
precision = precision_score(y_valid, valid_pred_m47)
recall = recall_score(y_valid, valid_pred_m47)
f1 = f1_score(y_valid, valid_pred_m47)
cm = confusion_matrix(y_valid, valid_pred_m47)

print("\n===== Model 47 =====")
print("Best iteration:", model_m47.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M40 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975945:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999375:+.6f}")
print(f"Precision : {precision - 0.989232:+.6f}")
print(f"Recall    : {recall - 0.885954:+.6f}")
print(f"F1-score  : {f1 - 0.934749:+.6f}")

print("\n===== M42 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.976749:+.6f}")
print(f"Precision : {precision - 0.988257:+.6f}")
print(f"Recall    : {recall - 0.882180:+.6f}")
print(f"F1-score  : {f1 - 0.932211:+.6f}")

print("\n===== M44 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.976510:+.6f}")
print(f"Precision : {precision - 0.985528:+.6f}")
print(f"Recall    : {recall - 0.885115:+.6f}")
print(f"F1-score  : {f1 - 0.932626:+.6f}")

Number of features: 11
Train: (907672, 11)
Valid: (389003, 11)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797

===== NaN count =====
category                           0
amt                                0
trans_hour                         0
age                                0
amt_to_prior_median_ratio       1649
rolling_sum_amt_1h                 0
recent_24h_high_amt_count          0
amt_zscore_card                    0
prior_normal_median_amt         1649
count_30min                        0
has_prior_normal_transaction       0
dtype: int64

has_prior_normal_transaction value counts:
has_prior_normal_transaction
1    1295026
0       1649
Name: count, dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.93216	valid_0's binary_logloss: 0.0291024
[100]	valid_0's average_precision: 0.958346	valid_0's binary_logloss: 0.0172944
[150]	valid_0's average_precision: 0.966447	valid_0's binary_logloss: 0.01233

M47은 큰 추가 이득은 없었음. has_prior_normal_transaction은 현재 조합에서는 우선순위를 낮춰도 될 듯.

그리고 has_prior_normal_transaction=0이 전체에서 1,649건뿐이라 이 변수가 모델 전체 성능을 크게 바꿀 기회 자체가 많지 않았음. 따라서 “쓸모없다”까지는 아니지만, M40에 굳이 추가할 만큼 명확한 이득은 관찰되지 않았다.

# Model 48 = M40 + merchant_change_count + high_speed

둘 다 M40에 단독 추가했을 때 PR-AUC가 올라갔고, 정보 성격도 다름

- merchant_change_count → 가맹점 변화 행동
- high_speed → 이동 이상성

그래서 이번에는 행동 변화 + 공간 이상 신호가 같이 있을 때 보완 효과가 생기는지 보는 실험.

In [39]:
# ============================================================
# Model 48
# M40 + merchant_change_count + high_speed
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m48 = [
    # M14
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",

    # M40 추가 변수
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",

    # M48 추가 변수
    "merchant_change_count",
    "high_speed"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m48]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m48]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Number of features:", len(features_m48))
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("\n===== NaN count =====")
print(df[features_m48].isna().sum())

model_m48 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m48.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m48 = model_m48.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m48 = (valid_prob_m48 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m48)
roc_auc = roc_auc_score(y_valid, valid_prob_m48)
precision = precision_score(y_valid, valid_pred_m48)
recall = recall_score(y_valid, valid_pred_m48)
f1 = f1_score(y_valid, valid_pred_m48)
cm = confusion_matrix(y_valid, valid_pred_m48)

print("\n===== Model 48 =====")
print("Best iteration:", model_m48.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M40 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975945:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999375:+.6f}")
print(f"Precision : {precision - 0.989232:+.6f}")
print(f"Recall    : {recall - 0.885954:+.6f}")
print(f"F1-score  : {f1 - 0.934749:+.6f}")

print("\n===== M42 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.976749:+.6f}")
print(f"Precision : {precision - 0.988257:+.6f}")
print(f"Recall    : {recall - 0.882180:+.6f}")
print(f"F1-score  : {f1 - 0.932211:+.6f}")

print("\n===== M44 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.976510:+.6f}")
print(f"Precision : {precision - 0.985528:+.6f}")
print(f"Recall    : {recall - 0.885115:+.6f}")
print(f"F1-score  : {f1 - 0.932626:+.6f}")

Number of features: 12
Train: (907672, 12)
Valid: (389003, 12)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797

===== NaN count =====
category                        0
amt                             0
trans_hour                      0
age                             0
amt_to_prior_median_ratio    1649
rolling_sum_amt_1h              0
recent_24h_high_amt_count       0
amt_zscore_card                 0
prior_normal_median_amt      1649
count_30min                     0
merchant_change_count           0
high_speed                      0
dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.920735	valid_0's binary_logloss: 0.0293665
[100]	valid_0's average_precision: 0.947828	valid_0's binary_logloss: 0.017557
[150]	valid_0's average_precision: 0.960863	valid_0's binary_logloss: 0.0123846
[200]	valid_0's average_precision: 0.965919	valid_0's binary_logloss: 0.00982525
[250]	valid_0's average_precision: 0.969

merchant_change_count와 high_speed는 각각 M40에 단독 추가했을 때는 PR-AUC를 올렸지만, 둘을 같이 넣으니까 오히려 전체 성능이 떨어짐. 그래서 현재 조건에서는 둘을 동시에 유지할 이유는 약함.

# Model 49 = M40 + merchant_change_count + is_high_amt

- merchant_change_count는 M42에서 PR-AUC를 최고 수준으로 올렸고
- is_high_amt는 M45에서 PR-AUC를 소폭 개선했음.

둘은 각각 가맹점 변화 행동과 고액거래 여부라는 다른 정보축이라, 같이 넣었을 때 보완 효과가 있는지 확인.

In [40]:
# ============================================================
# Model 49
# M40 + merchant_change_count + is_high_amt
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m49 = [
    # M14
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",

    # M40 추가 변수
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",

    # M49 추가 변수
    "merchant_change_count",
    "is_high_amt"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m49]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m49]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Number of features:", len(features_m49))
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("\n===== NaN count =====")
print(df[features_m49].isna().sum())

model_m49 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m49.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m49 = model_m49.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m49 = (valid_prob_m49 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m49)
roc_auc = roc_auc_score(y_valid, valid_prob_m49)
precision = precision_score(y_valid, valid_pred_m49)
recall = recall_score(y_valid, valid_pred_m49)
f1 = f1_score(y_valid, valid_pred_m49)
cm = confusion_matrix(y_valid, valid_pred_m49)

print("\n===== Model 49 =====")
print("Best iteration:", model_m49.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M40 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975945:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999375:+.6f}")
print(f"Precision : {precision - 0.989232:+.6f}")
print(f"Recall    : {recall - 0.885954:+.6f}")
print(f"F1-score  : {f1 - 0.934749:+.6f}")

print("\n===== M42 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.976749:+.6f}")
print(f"Precision : {precision - 0.988257:+.6f}")
print(f"Recall    : {recall - 0.882180:+.6f}")
print(f"F1-score  : {f1 - 0.932211:+.6f}")

print("\n===== M45 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.976175:+.6f}")
print(f"Precision : {precision - 0.987816:+.6f}")
print(f"Recall    : {recall - 0.883857:+.6f}")
print(f"F1-score  : {f1 - 0.932950:+.6f}")

print("\n===== M48 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.974018:+.6f}")
print(f"Precision : {precision - 0.984060:+.6f}")
print(f"Recall    : {recall - 0.880084:+.6f}")
print(f"F1-score  : {f1 - 0.929172:+.6f}")

Number of features: 12
Train: (907672, 12)
Valid: (389003, 12)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797

===== NaN count =====
category                        0
amt                             0
trans_hour                      0
age                             0
amt_to_prior_median_ratio    1649
rolling_sum_amt_1h              0
recent_24h_high_amt_count       0
amt_zscore_card                 0
prior_normal_median_amt      1649
count_30min                     0
merchant_change_count           0
is_high_amt                     0
dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.931594	valid_0's binary_logloss: 0.0291831
[100]	valid_0's average_precision: 0.95404	valid_0's binary_logloss: 0.0176456
[150]	valid_0's average_precision: 0.964495	valid_0's binary_logloss: 0.0126759
[200]	valid_0's average_precision: 0.967705	valid_0's binary_logloss: 0.0101336
[250]	valid_0's average_precision: 0.9694

M49는 확실히 M40/M42를 못 넘음. 그래서 merchant_change_count + is_high_amt 동시 추가도 현재 조건에서는 이득이 없다고 보면 됨.

# Model 50 = M40 + merchant_change_count + risk_time_22_04 + is_online, 총 13개 변수.

행동 변화 + 시간대 + 거래 채널을 한 번에 보완했을 때 M40보다 랭킹 성능이 좋아지는지 확인하는 실험

In [41]:
# ============================================================
# Model 50
# M40 + merchant_change_count + risk_time_22_04 + is_online
# 총 13개 변수
# ============================================================

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

features_m50 = [
    # M40
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "recent_24h_high_amt_count",
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",

    # M50 추가
    "merchant_change_count",
    "risk_time_22_04",
    "is_online"
]

split_idx = int(len(df) * 0.7)

X_train = df.loc[:split_idx-1, features_m50]
y_train = df.loc[:split_idx-1, "is_fraud"]

X_valid = df.loc[split_idx:, features_m50]
y_valid = df.loc[split_idx:, "is_fraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Number of features:", len(features_m50))
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Train fraud:", pos)
print("Valid fraud:", int(y_valid.sum()))
print("scale_pos_weight:", scale_pos_weight)

print("\n===== NaN count =====")
print(df[features_m50].isna().sum())

model_m50 = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_m50.fit(
    X_train,
    y_train,
    eval_X=X_valid,
    eval_y=y_valid,
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

valid_prob_m50 = model_m50.predict_proba(X_valid)[:, 1]

threshold = 0.990239
valid_pred_m50 = (valid_prob_m50 >= threshold).astype("int8")

pr_auc = average_precision_score(y_valid, valid_prob_m50)
roc_auc = roc_auc_score(y_valid, valid_prob_m50)
precision = precision_score(y_valid, valid_pred_m50)
recall = recall_score(y_valid, valid_pred_m50)
f1 = f1_score(y_valid, valid_pred_m50)
cm = confusion_matrix(y_valid, valid_pred_m50)

print("\n===== Model 50 =====")
print("Best iteration:", model_m50.best_iteration_)
print(f"PR-AUC    : {pr_auc:.6f}")
print(f"ROC-AUC   : {roc_auc:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-score  : {f1:.6f}")
print("Confusion Matrix:")
print(cm)

print("\n===== M40 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.975945:+.6f}")
print(f"ROC-AUC   : {roc_auc - 0.999375:+.6f}")
print(f"Precision : {precision - 0.989232:+.6f}")
print(f"Recall    : {recall - 0.885954:+.6f}")
print(f"F1-score  : {f1 - 0.934749:+.6f}")

print("\n===== M42 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.976749:+.6f}")
print(f"Precision : {precision - 0.988257:+.6f}")
print(f"Recall    : {recall - 0.882180:+.6f}")
print(f"F1-score  : {f1 - 0.932211:+.6f}")

print("\n===== M44 대비 =====")
print(f"PR-AUC    : {pr_auc - 0.976510:+.6f}")
print(f"Precision : {precision - 0.985528:+.6f}")
print(f"Recall    : {recall - 0.885115:+.6f}")
print(f"F1-score  : {f1 - 0.932626:+.6f}")


Number of features: 13
Train: (907672, 13)
Valid: (389003, 13)
Train fraud: 5121
Valid fraud: 2385
scale_pos_weight: 176.24506932239797

===== NaN count =====
category                        0
amt                             0
trans_hour                      0
age                             0
amt_to_prior_median_ratio    1649
rolling_sum_amt_1h              0
recent_24h_high_amt_count       0
amt_zscore_card                 0
prior_normal_median_amt      1649
count_30min                     0
merchant_change_count           0
risk_time_22_04                 0
is_online                       0
dtype: int64
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.945618	valid_0's binary_logloss: 0.0280285
[100]	valid_0's average_precision: 0.961374	valid_0's binary_logloss: 0.0169095
[150]	valid_0's average_precision: 0.96624	valid_0's binary_logloss: 0.0122801
[200]	valid_0's average_precision: 0.968629	valid_0's binary_logloss: 0.00970583
[250]	